# v29 — v25 분류기 축소와 재현 가능한 단순화

전처리·변수 표현·각 분류기 하이퍼파라미터는 유지하고 분류기를 제거합니다. 남은 기존 혼합 비중만 정규화하며 새 가중치 탐색은 하지 않습니다. 3-fold 선택에서 14개 모델의 개별 제외 효과, 핵심 3종의 모든 1~3개 조합, greedy 제거를 비교합니다. 허용 손실 내 최소 모델 수를 고정한 뒤 다른 seed의 그룹 5-fold에서 새로 학습해 확인합니다.

선택과 확인은 서로 다른 분할이지만 같은 과거 개발 환자를 사용하므로 외부 독립 검증은 아닙니다. test는 마지막 고정 모델 추론에만 사용합니다. Geneformer와 형제 라벨 규칙은 포함하지 않습니다. 모든 구성 정의·선별·학습·저장·추론이 이 노트북에 들어 있습니다. worker 파일은 이 노트북이 artifacts 아래 자동 생성하며 외부 프로젝트 Python 파일이 필요하지 않습니다.


In [1]:
COMPONENT_SOURCE29=["import os,sys,re,json,pickle,cloudpickle,hashlib,platform,time,copy,warnings\nfrom pathlib import Path\nfrom collections import Counter\nimport numpy as np\nimport pandas as pd\nfrom scipy import sparse\nfrom scipy.special import softmax\nfrom sklearn.base import BaseEstimator,TransformerMixin,clone\nfrom sklearn.feature_extraction import DictVectorizer\nfrom sklearn.feature_selection import chi2,f_classif\nfrom sklearn.preprocessing import StandardScaler,MaxAbsScaler\nfrom sklearn.multiclass import OneVsRestClassifier\nfrom sklearn.model_selection import StratifiedGroupKFold\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.ensemble import ExtraTreesClassifier\nfrom sklearn.metrics import f1_score,accuracy_score,classification_report,confusion_matrix\nfrom lightgbm import LGBMClassifier\nfrom threadpoolctl import threadpool_limits\nfrom IPython.display import display,FileLink\nthreadpool_limits(2)\nSEED=42\nforbidden_attempts23=[]\ndef forbid_inference23(event,args):\n if event=='open' and args and isinstance(args[0],(str,bytes,os.PathLike)) and Path(os.fsdecode(args[0])).name in {'test.csv','sample_submission.csv'}:\n  forbidden_attempts23.append(str(args[0]));raise RuntimeError('Inference data access is forbidden in v23')\nsys.addaudithook(forbid_inference23)\nroots=[Path.cwd(),*Path.cwd().parents]\nREPO=next((p for p in roots if (p/'data/train.csv').is_file()),None)\nif REPO is None:raise FileNotFoundError('data/train.csv가 필요합니다.')\nDATA=REPO/'data';WORK=REPO/'models'/'혜진' if (REPO/'models'/'혜진').is_dir() else REPO\nOUT=WORK/'artifacts'/'v29'/'definitions_v25';OUT.mkdir(parents=True,exist_ok=True)\n# Inference files and submission directory are intentionally not used.\nos.environ['MPLCONFIGDIR']=str(OUT/'.mplconfig')\ntrain=pd.read_csv(DATA/'train.csv')\ngenes=[c for c in train if c not in ['ID','SUBCLASS']]\nassert train.ID.is_unique\ny=train.SUBCLASS.to_numpy();labels=np.sort(np.unique(y));yi=np.searchsorted(labels,y);NC=len(labels)\n\ndef normalize_token25(token):\n return re.sub(r'^(p\\.)?([ACDEFGHIKLMNPQRSTVWY])([1-9][0-9]*)X$',lambda m:(m[1] or '')+m[2]+m[3]+'*',token)\ndef canonical_frame25(frame):\n def one(v):\n  if not isinstance(v,str) or 'X' not in v:return v\n  return ' '.join(normalize_token25(t) for t in v.split())\n return frame.map(one)\ndef legacy_frame25(frame,mixed=False):\n def one(v):\n  if not isinstance(v,str) or '*' not in v:return v\n  def conv(m):\n   token=m[0]\n   if mixed and int(hashlib.sha256(token.encode()).hexdigest()[:8],16)%2:return token\n   return token[:-1]+'X'\n  return re.sub(r'(?<!\\S)(?:p\\.)?[ACDEFGHIKLMNPQRSTVWY][1-9][0-9]*\\*(?!\\S)',conv,v)\n return frame.map(one)\n\nX25_original=train[genes].copy();X25_mixed=legacy_frame25(X25_original,mixed=True);X=canonical_frame25(X25_mixed);assert X.equals(X25_original);groups=pd.util.hash_pandas_object(X,index=False).to_numpy()\ndev,hold=next(StratifiedGroupKFold(5,shuffle=True,random_state=SEED).split(X,y,groups))\nfolds=list(StratifiedGroupKFold(3,shuffle=True,random_state=SEED+1).split(X.iloc[dev],y[dev],groups[dev]))\nassert not set(groups[dev]).intersection(groups[hold])\nfor a,b in folds:assert not set(groups[dev[a]]).intersection(groups[dev[b]])\npd.DataFrame({'ID':train.ID,'split':np.where(np.isin(np.arange(len(train)),hold),'holdout','development'),'group':groups.astype(str)}).to_csv(OUT/'split_manifest.csv',index=False)\nprint('TRAIN ONLY',train.shape,'Dev/holdout',len(dev),len(hold),'OUT',OUT,flush=True)\n\nfrom sklearn.neural_network import MLPClassifier\nfrom sklearn.feature_extraction.text import TfidfTransformer\nfrom sklearn.utils.class_weight import compute_sample_weight\nfrom xgboost import XGBClassifier\n", "GROUPS = {\n 'p53_cellcycle': 'TP53 RB1 CDKN2A CDKN2B CDK4 CDK6 CCND1 ATM CHEK2'.split(),\n 'rtk_ras_mapk': 'EGFR ERBB2 ERBB3 KRAS NRAS HRAS BRAF NF1 MAP2K1 MAP2K2 KIT PDGFRA FGFR2 FGFR3'.split(),\n 'pi3k_mtor': 'PIK3CA PIK3R1 PTEN AKT1 MTOR TSC1 TSC2 STK11'.split(),\n 'dna_repair': 'BRCA1 BRCA2 PALB2 ATM ATR CHEK2 MLH1 MSH2 MSH6 PMS2 POLE POLD1'.split(),\n 'chromatin': 'ARID1A ARID1B ARID2 PBRM1 SMARCA4 BAP1 SETD2 KMT2A KMT2C KMT2D ATRX'.split(),\n 'wnt': 'APC CTNNB1 AXIN1 AXIN2 RNF43'.split(),\n 'metabolism': 'IDH1 IDH2 SDHB SDHC SDHD FH VHL'.split(),\n}\n\ndef kind(token):\n if 'fs' in token.lower(): return 'frameshift'\n if '*' in token: return 'stop'\n if 'del' in token.lower(): return 'deletion'\n if 'ins' in token.lower(): return 'insertion'\n m = re.fullmatch(r'([A-Z])(\\d+)([A-Z])', token)\n if m: return 'synonymous' if m[1] == m[3] else 'substitution'\n return 'other'\n\nclass MutationFeatures(BaseEstimator, TransformerMixin):\n def __init__(self, mode='full', min_df=3):\n  self.mode = mode\n  self.min_df = min_df\n\n def dictionaries(self, X):\n  genes = list(X.columns)\n  for row in X.itertuples(index=False, name=None):\n   d = {}; counts = {}; burden = 0; events = 0; positions = []\n   changed = set()\n   for gene, value in zip(genes, row):\n    if value is None or str(value) in ('nan', '', '<NA>'):\n     d['missing:' + gene] = 1; continue\n    if value == 'WT': continue\n    changed.add(gene); burden += 1; d['gene:' + gene] = 1\n    if self.mode == 'binary': continue\n    tokens = str(value).split()\n    for raw_token in tokens:\n     token = re.sub(r'^([A-Z]\\d+)X$', r'\\1*', raw_token)\n     events += 1; typ = kind(token); counts[typ] = counts.get(typ, 0) + 1\n     d['variant:' + gene + ':' + token] = 1\n     d['type:' + gene + ':' + typ] = 1\n     m = re.search(r'\\d+', token)\n     if m:\n      position = int(m[0]); positions.append(position)\n      d['position:' + gene + ':' + str(position)] = 1\n   if self.mode == 'full':\n    d['numeric:log_mutated_genes'] = np.log1p(burden)\n    d['numeric:log_events'] = np.log1p(events)\n    d['numeric:multi_event_excess'] = np.log1p(max(0, events-burden))\n    for typ in ['frameshift','stop','deletion','insertion','synonymous','substitution','other']:\n     d['numeric:log_' + typ] = np.log1p(counts.get(typ,0))\n     d['numeric:fraction_' + typ] = counts.get(typ,0)/max(events,1)\n    d['numeric:log_mean_position'] = np.log1p(np.mean(positions)) if positions else 0.\n    for name, members in GROUPS.items():\n     d['module:' + name] = np.log1p(len(changed.intersection(members)))\n   yield d\n\n def fit(self, X, y=None):\n  self.genes_ = list(X.columns)\n  self.vectorizer_ = DictVectorizer(dtype=np.float32)\n  matrix = self.vectorizer_.fit_transform(self.dictionaries(X))\n  names = self.vectorizer_.get_feature_names_out()\n  self.keep_ = (np.asarray((matrix != 0).sum(axis=0)).ravel() >= self.min_df) | np.array([s.startswith(('numeric:', 'module:')) for s in names])\n  self.feature_names_ = names[self.keep_]\n  return self\n\n def transform(self, X):\n  missing = set(self.genes_) - set(X.columns)\n  if missing: raise ValueError('Missing gene columns: ' + str(sorted(missing)[:10]))\n  return sparse.csr_matrix(self.vectorizer_.transform(self.dictionaries(X[self.genes_]))[:, self.keep_])\n\n def get_feature_names_out(self, input_features=None): return self.feature_names_\ndef canon(t):\n t=t.strip().removeprefix('p.');t=re.sub(r'(?<=\\d)X$','*',t)\n return re.sub(r'^([A-Z]\\d+)(?:[A-Z*]+)?fs.*$',r'\\1fs',t)\ndef aa_change(t):\n m=re.fullmatch(r'([A-Z])\\d+([A-Z*])',t)\n if m:return m[1]+'>'+m[2]\n m=re.match(r'([A-Z])\\d+',t)\n if m:\n  for suffix in ['fs','del','ins']:\n   if suffix in t:return m[1]+'>'+suffix\n return None\n\ndef parse_events(frame):\n names=list(frame.columns);result=[]\n for row in frame.itertuples(index=False,name=None):\n  result.append([(g,tuple(sorted(set(canon(t) for t in v.split())))) for g,v in zip(names,row) if isinstance(v,str) and v!='WT'])\n return result\n\nclass EvidenceEncoder:\n def __init__(self,unknown=1.,silent=1.,support=2):self.unknown=unknown;self.silent=silent;self.support=support\n def fit(self,events):\n  self.token_df={};self.change_df={}\n  if self.unknown<1:\n   tokens=Counter();changes=Counter()\n   for row in events:\n    for g,ts in row:\n     tokens.update((g,t) for t in ts)\n     changes.update(set((g,a) for t in ts if (a:=aa_change(t)) is not None))\n   self.token_df={k:v for k,v in tokens.items() if v>=self.support}\n   self.change_df={k:v for k,v in changes.items() if v>=self.support}\n  return self\n def transform(self,events,training=False,exclude_events=None):\n  result=[]\n  for row_number,row in enumerate(events):\n   own=exclude_events[row_number] if exclude_events is not None else (row if training else [])\n   own_tokens={(gg,tt) for gg,tts in own for tt in tts}\n   own_changes={(gg,ac) for gg,tts in own for tt in tts if (ac:=aa_change(tt)) is not None}\n   d={};gene_weights={};type_sums=Counter();non_syn=0.;trunc=0.;multikind=0\n   for g,ts in row:\n    types={};gene=0.;known_gene=False\n    for t in ts:\n     typ=kind(t);a=aa_change(t)\n     known=self.token_df.get((g,t),0)-int((g,t) in own_tokens)>=self.support\n     known_aa=a is not None and self.change_df.get((g,a),0)-int((g,a) in own_changes)>=self.support\n     if typ=='synonymous':w=self.silent\n     elif known or typ in ['stop','frameshift','deletion','insertion']:w=1.\n     elif known_aa:w=max(self.unknown,.75)\n     else:w=self.unknown\n     if w<=0:continue\n     gene=max(gene,w);types[typ]=max(types.get(typ,0),w)\n     d['variant:'+g+':'+t]=max(d.get('variant:'+g+':'+t,0),w)\n     m=re.search(r'\\d+',t)\n     if m:d['position:'+g+':'+m[0]]=max(d.get('position:'+g+':'+m[0],0),w)\n     if a:d['aa:'+g+':'+a]=max(d.get('aa:'+g+':'+a,0),w)\n     if known and typ!='synonymous':known_gene=True\n    if gene<=0:continue\n    d['gene:'+g]=gene;gene_weights[g]=gene\n    for typ,w in types.items():d['type:'+g+':'+typ]=w;type_sums[typ]+=w\n    if known_gene:d['supported:'+g]=1.\n    non_syn+=max([w for k,w in types.items() if k!='synonymous'] or [0.])\n    trunc+=max([w for k,w in types.items() if k in ['stop','frameshift','deletion','insertion']] or [0.])\n    multikind+=len(types)>1\n   n=len(row)\n   d['numeric:log_raw_genes']=np.log1p(n)\n   d['numeric:log_effective_genes']=np.log1p(sum(gene_weights.values()))\n   d['numeric:log_nonsyn_genes']=np.log1p(non_syn)\n   d['numeric:log_trunc_genes']=np.log1p(trunc)\n   d['numeric:log_multikind_genes']=np.log1p(multikind)\n   for typ in ['synonymous','substitution','stop','frameshift','deletion','insertion','other']:\n    d['numeric:gene_fraction_'+typ]=type_sums[typ]/max(n,1)\n   for name,gs in GROUPS.items():d['module:'+name]=np.log1p(sum(gene_weights.get(g,0) for g in gs))\n   result.append(d)\n  return result\n\nevents=parse_events(X)\nlegacy_records=list(MutationFeatures(mode='full').dictionaries(X))\nassert canon('R213X')=='R213*' and canon('K16Rfs')=='K16fs'\nassert aa_change('R132H')==aa_change('R140H')=='R>H'\nprint('Parsed rows:',len(events),'Normalization examples passed')\n", "def relevance_scores(a,target,method):\n base=method.removesuffix('_full')\n if base=='embedded':\n  scaled=csr32(MaxAbsScaler().fit_transform(a))\n  fitted=OneVsRestClassifier(LogisticRegression(solver='liblinear',l1_ratio=1,C=.2,class_weight='balanced',max_iter=2000),n_jobs=1).fit(scaled,target)\n  return np.max(np.abs(np.vstack([e.coef_[0] for e in fitted.estimators_])),axis=0)\n return (chi2(a,target) if base=='chi2' else f_classif(a,target))[0]\n\ndef csr32(a):\n b=sparse.csr_matrix(a,dtype=np.float32)\n if b.nnz>=np.iinfo(np.int32).max:raise ValueError('Sparse matrix too large for tree index type')\n b.indices=b.indices.astype(np.int32,copy=False);b.indptr=b.indptr.astype(np.int32,copy=False)\n return b\n\nclass Space:\n def fit(self,rs):\n  self.vec=DictVectorizer(dtype=np.float32);a=self.vec.fit_transform(rs)\n  names=self.vec.get_feature_names_out();df=np.asarray((a!=0).sum(axis=0)).ravel()\n  keep=(df>=3)|np.array([n.startswith(('numeric:','module:')) for n in names])\n  self.raw=np.flatnonzero(keep);self.names=names[keep]\n  return csr32(a[:,self.raw])\n def transform(self,rs):return csr32(self.vec.transform(rs)[:,self.raw])\n def compact_vec(self,indices):\n  v=copy.deepcopy(self.vec);mask=np.zeros(len(v.feature_names_),bool);mask[self.raw[np.sort(indices)]]=True\n  v.restrict(mask);return v\n\nclass Selector:\n def __init__(self,a,target,names):\n  self.a=a;self.y=target;self.names=names;self.cache={};self.corr={};self.scores={};self.audit=[];self.examples=[]\n def order(self,method,threshold):\n  key=(method,threshold)\n  if key in self.cache:return self.cache[key]\n  if method not in self.scores:\n   with warnings.catch_warnings():\n    warnings.simplefilter('ignore',RuntimeWarning);score=relevance_scores(self.a,self.y,method)\n   score=np.nan_to_num(score,nan=0,posinf=np.finfo(np.float32).max,neginf=0)\n   if not method.endswith('_full'):score[np.array([n.startswith('numeric:') for n in self.names])]=0\n   self.scores[method]=score\n  score=self.scores[method]\n  if method not in self.corr:\n   pool=np.argsort(-score,kind='stable')[:min(4096,int((score>0).sum()))]\n   z=self.a[:,pool];mean=np.asarray(z.mean(axis=0)).ravel();var=np.asarray(z.multiply(z).mean(axis=0)).ravel()-mean**2\n   sd=np.sqrt(np.maximum(var,1e-12));corr=(z.T@z).toarray()/z.shape[0]-np.outer(mean,mean)\n   corr/=np.outer(sd,sd);np.clip(corr,-1,1,out=corr)\n   self.corr[method]=(pool,corr)\n  pool,corr=self.corr[method];chosen=[]\n  for i in range(len(pool)):\n   if chosen:\n    correlations=np.abs(corr[i,chosen]);j=int(np.argmax(correlations))\n    if correlations[j]>=threshold:\n     if len(self.examples)<100:self.examples.append({'method':method,'threshold':threshold,'kept':self.names[pool[chosen[j]]],'discarded':self.names[pool[i]],'correlation':float(corr[i,chosen[j]])})\n     continue\n   chosen.append(i)\n   if len(chosen)>=2048:break\n  order=pool[chosen];self.cache[key]=order\n  self.audit.append({'method':method,'threshold':threshold,'association_pool':len(pool),'remaining_up_to_2048':len(order)})\n  return order\n def select(self,method,threshold,k):return np.sort(self.order(method,threshold)[:k])\n\nclass CompactPredictor:\n def __init__(self,space,cols,est):\n  self.vec=space.compact_vec(cols);self.names=self.vec.get_feature_names_out();self.est=est\n  self.direct_inputs=len(self.names);self.raw_inputs=len(self.names);self.kind='compact'\n def predict(self,rs):return self.est.predict_proba(csr32(self.vec.transform(rs)))\n\ndef background_frame(frame,training_frame,seed,count):\n rng=np.random.default_rng(seed);out=frame.copy();arr=out.to_numpy(copy=True)\n rate=(training_frame.notna()&training_frame.ne('WT')).mean().to_numpy();prob=rate/rate.sum();idx=np.flatnonzero(prob>0)\n for r in range(len(arr)):\n  n=min(len(idx),int(rng.poisson(count)))\n  for c in rng.choice(len(prob),size=n,replace=False,p=prob):\n   token='A1A' if rng.random()<.5 else 'A1V'\n   arr[r,c]=token if not isinstance(arr[r,c],str) or arr[r,c]=='WT' else arr[r,c]+' '+token\n return pd.DataFrame(arr,columns=frame.columns,index=frame.index)\nclass ColumnPost:\n def __init__(self,mode='none'):self.mode=mode\n def fit(self,a,names):\n  self.raw_names=np.array(names);self.names=np.array(names)\n  if self.mode=='maxabs':self.scaler=MaxAbsScaler().fit(a)\n  if self.mode=='block':\n   self.binary=np.flatnonzero([not n.startswith(('numeric:','module:')) for n in names])\n   self.summary=np.flatnonzero([n.startswith(('numeric:','module:')) for n in names])\n   self.tf=TfidfTransformer(norm='l2',use_idf=True).fit(a[:,self.binary])\n   self.scaler=MaxAbsScaler().fit(a[:,self.summary])\n   self.names=np.concatenate([self.raw_names[self.binary],self.raw_names[self.summary]])\n  return self\n def transform(self,a):\n  if self.mode=='maxabs':return csr32(self.scaler.transform(a))\n  if self.mode=='block':return csr32(sparse.hstack([self.tf.transform(a[:,self.binary]),self.scaler.transform(a[:,self.summary])],format='csr'))\n  return csr32(a)\n\nclass Member:\n def __init__(self,name,view,encoder,space,cols,post,est):\n  self.name=name;self.view=view;self.encoder=encoder;self.vec=space.compact_vec(cols);self.post=post;self.est=est\n  self.raw_names=self.vec.get_feature_names_out();self.names=post.names\n def predict_rows(self,rs):return self.est.predict_proba(self.post.transform(csr32(self.vec.transform(rs))))\n\nVIEWS={'raw':(1.,1.),'syn_down':(1.,.25),'gated':(.25,1.),'gated_syn_down':(.25,.25),'gated_nonsyn':(.25,0.)}\nCONFIGS={}\nfor view in VIEWS:\n for k in [1024,2048]:\n  for c in [.1,.5]:CONFIGS[f'{view}_lr_k{k}_C{c}']=dict(view=view,algo='lr',k=k,C=c,post='none',method='chi2_full')\n  CONFIGS[f'{view}_lgb_k{k}']=dict(view=view,algo='lgb',k=k,post='none',method='f_classif_full')\nfor view in ['raw','gated_syn_down']:\n for depth in [3,5]:CONFIGS[f'{view}_xgb_d{depth}']=dict(view=view,algo='xgb',k=2048,depth=depth,post='none',method='f_classif_full')\n for alpha in [1,10]:CONFIGS[f'{view}_mlp_a{alpha}']=dict(view=view,algo='mlp',k=1024,alpha=alpha,post='maxabs',method='chi2_full')\n for c in [1,4]:CONFIGS[f'{view}_blocklr_C{c}']=dict(view=view,algo='lr',k=2048,C=c,post='block',method='chi2_full')\n CONFIGS[f'{view}_extra']=dict(view=view,algo='extra',k=1024,post='none',method='f_classif_full')\n(OUT/'configurations.json').write_text(json.dumps(CONFIGS,indent=2))\n\ndef estimator(cfg):\n algo=cfg['algo']\n if algo=='lr':return LogisticRegression(C=cfg['C'],class_weight='balanced',max_iter=2000)\n if algo=='lgb':return LGBMClassifier(n_estimators=300,learning_rate=.04,num_leaves=15,min_child_samples=20,reg_lambda=10,colsample_bytree=.85,class_weight='balanced',verbosity=-1,n_jobs=2,random_state=SEED)\n if algo=='xgb':return XGBClassifier(n_estimators=300,max_depth=cfg['depth'],learning_rate=.04,min_child_weight=3,reg_lambda=10,subsample=.85,colsample_bytree=.85,max_bin=63,tree_method='hist',objective='multi:softprob',eval_metric='mlogloss',n_jobs=2,random_state=SEED)\n if algo=='mlp':return MLPClassifier(hidden_layer_sizes=(128,),activation='relu',alpha=cfg['alpha'],learning_rate_init=.001,batch_size=128,max_iter=250,tol=1e-4,n_iter_no_change=15,early_stopping=False,random_state=SEED)\n return ExtraTreesClassifier(n_estimators=250,max_depth=20,min_samples_leaf=3,max_features='sqrt',class_weight='balanced',n_jobs=2,random_state=SEED)\n\ndef prepare_view(idx,view):\n ev=[events[i] for i in idx];enc=EvidenceEncoder(*VIEWS[view]).fit(ev)\n rs=enc.transform(ev,training=True);space=Space();a=space.fit(rs);selector=Selector(a,yi[idx],space.names)\n return enc,space,a,selector\n\ndef train_new(name,idx,cache):\n cfg=CONFIGS[name];view=cfg['view']\n if view not in cache:cache[view]=prepare_view(idx,view)\n enc,space,a,selector=cache[view];cols=selector.select(cfg['method'],.98,cfg['k'])\n post=ColumnPost(cfg['post']).fit(a[:,cols],space.names[cols]);tx=post.transform(a[:,cols]);target=yi[idx]\n row_weights=np.ones(len(idx),dtype=float)\n if cfg.get('augment',False):\n  if '_aug_events' not in cache:\n   cache['_aug_events']=parse_events(rare_background(X.iloc[idx],[events[i] for i in idx],SEED+777,count=10))\n  aug_rs=enc.transform(cache['_aug_events'],exclude_events=[events[i] for i in idx])\n  aug_x=post.transform(space.transform(aug_rs)[:,cols]);tx=csr32(sparse.vstack([tx,aug_x],format='csr'))\n  target=np.concatenate([target,target]);row_weights=np.concatenate([row_weights,.5*row_weights])\n est=estimator(cfg)\n if cfg['algo'] in ['xgb','mlp']:est.fit(tx,target,sample_weight=compute_sample_weight('balanced',target)*row_weights)\n elif cfg.get('augment',False):est.fit(tx,target,sample_weight=row_weights)\n else:est.fit(tx,target)\n return Member(name,view,enc,space,cols,post,est)\n\nLEGACY={\n 'legacy_lr':dict(k=2048,method='chi2',algo='lr'),\n 'legacy_lgb':dict(k=2048,method='f_classif_full',algo='lgb'),\n 'legacy_extra':dict(k=1024,method='f_classif_full',algo='extra')}\ndef train_legacy(idx):\n space=Space();a=space.fit([legacy_records[i] for i in idx]);sel=Selector(a,yi[idx],space.names);result=[]\n for name,cfg in LEGACY.items():\n  cols=sel.select(cfg['method'],.95,cfg['k'])\n  if cfg['algo']=='lr':est=LogisticRegression(C=.1,class_weight='balanced',max_iter=2000)\n  elif cfg['algo']=='lgb':est=LGBMClassifier(n_estimators=250,learning_rate=.05,num_leaves=15,min_child_samples=20,reg_lambda=10,colsample_bytree=.85,class_weight='balanced',verbosity=-1,n_jobs=2,random_state=SEED)\n  else:est=ExtraTreesClassifier(n_estimators=250,max_depth=20,min_samples_leaf=3,max_features='sqrt',class_weight='balanced',n_jobs=2,random_state=SEED)\n  est.fit(a[:,cols],yi[idx]);post=ColumnPost().fit(a[:,cols],space.names[cols]);result.append(Member(name,'legacy',None,space,cols,post,est))\n return result\n\ndef duplicate_frame(frame):\n arr=frame.to_numpy(copy=True)\n for i,row in enumerate(arr):\n  for j,v in enumerate(row):\n   if isinstance(v,str) and v!='WT':arr[i,j]=v+' '+v\n return pd.DataFrame(arr,columns=frame.columns,index=frame.index)\n\ndef rare_background(frame,tr_events,seed,count=20):\n rng=np.random.default_rng(seed);df=Counter((g,t) for row in tr_events for g,ts in row for t in ts)\n syn=[];sub=[]\n for (g,t),n in df.items():\n  if n>2:continue\n  if kind(t)=='synonymous':syn.append((g,t))\n  elif kind(t)=='substitution':sub.append((g,t))\n arr=frame.to_numpy(copy=True);lookup={g:i for i,g in enumerate(frame.columns)}\n for row in arr:\n  for _ in range(int(rng.poisson(count))):\n   pool=syn if rng.random()<.5 else sub\n   g,t=pool[int(rng.integers(len(pool)))];j=lookup[g]\n   row[j]=t if not isinstance(row[j],str) or row[j]=='WT' else row[j]+' '+t\n return pd.DataFrame(arr,columns=frame.columns,index=frame.index)\n\ndef eval_views(idx,tr_idx,seed):\n clean=X.iloc[idx];noise=rare_background(clean,[events[i] for i in tr_idx],seed);dup=duplicate_frame(clean)\n frames=[clean,noise,dup]\n return frames,[parse_events(f) for f in frames],[list(MutationFeatures(mode='full').dictionaries(f)) for f in frames]\n\ndef predict_members(members,weights,frame):\n ev=None;old=None;cache={};p=None\n for m,w in zip(members,weights):\n  if m.view=='legacy':\n   if old is None:old=list(MutationFeatures(mode='full').dictionaries(frame[genes]))\n   rs=old\n  else:\n   if ev is None:ev=parse_events(frame[genes])\n   if m.view not in cache:cache[m.view]=m.encoder.transform(ev)\n   rs=cache[m.view]\n  q=m.predict_rows(rs);p=q*w if p is None else p+q*w\n return p\nprint('Novel settings:',len(CONFIGS),'Views:',VIEWS)\n\n# CV-guided targeted refinement: regularization, support threshold, and rare-variant augmentation.\nVIEWS['gated_syn_down_s3']=(.25,.25,3)\nCONFIGS.update({'raw_mlp_a0.01': {'view': 'raw', 'algo': 'mlp', 'k': 1024, 'alpha': 0.01, 'post': 'maxabs', 'method': 'chi2_full'}, 'raw_mlp_a0.1': {'view': 'raw', 'algo': 'mlp', 'k': 1024, 'alpha': 0.1, 'post': 'maxabs', 'method': 'chi2_full'}, 'gated_syn_down_mlp_a0.01': {'view': 'gated_syn_down', 'algo': 'mlp', 'k': 1024, 'alpha': 0.01, 'post': 'maxabs', 'method': 'chi2_full'}, 'gated_syn_down_mlp_a0.1': {'view': 'gated_syn_down', 'algo': 'mlp', 'k': 1024, 'alpha': 0.1, 'post': 'maxabs', 'method': 'chi2_full'}, 'gated_syn_down_s3_lr_k1024_C0.1': {'view': 'gated_syn_down_s3', 'algo': 'lr', 'k': 1024, 'C': 0.1, 'post': 'none', 'method': 'chi2_full'}, 'gated_syn_down_s3_lr_k1024_C0.5': {'view': 'gated_syn_down_s3', 'algo': 'lr', 'k': 1024, 'C': 0.5, 'post': 'none', 'method': 'chi2_full'}, 'gated_syn_down_s3_lgb_k1024': {'view': 'gated_syn_down_s3', 'algo': 'lgb', 'k': 1024, 'post': 'none', 'method': 'f_classif_full'}, 'gated_syn_down_s3_lr_k2048_C0.1': {'view': 'gated_syn_down_s3', 'algo': 'lr', 'k': 2048, 'C': 0.1, 'post': 'none', 'method': 'chi2_full'}, 'gated_syn_down_s3_lr_k2048_C0.5': {'view': 'gated_syn_down_s3', 'algo': 'lr', 'k': 2048, 'C': 0.5, 'post': 'none', 'method': 'chi2_full'}, 'gated_syn_down_s3_lgb_k2048': {'view': 'gated_syn_down_s3', 'algo': 'lgb', 'k': 2048, 'post': 'none', 'method': 'f_classif_full'}, 'gated_syn_down_s3_xgb_d3': {'view': 'gated_syn_down_s3', 'algo': 'xgb', 'k': 2048, 'depth': 3, 'post': 'none', 'method': 'f_classif_full'}, 'gated_syn_down_s3_xgb_d5': {'view': 'gated_syn_down_s3', 'algo': 'xgb', 'k': 2048, 'depth': 5, 'post': 'none', 'method': 'f_classif_full'}, 'raw_aug_lgb': {'view': 'raw', 'algo': 'lgb', 'k': 2048, 'post': 'none', 'method': 'f_classif_full', 'augment': True}, 'raw_aug_xgb_d5': {'view': 'raw', 'algo': 'xgb', 'k': 2048, 'depth': 5, 'post': 'none', 'method': 'f_classif_full', 'augment': True}, 'gated_s3_aug_lgb': {'view': 'gated_syn_down_s3', 'algo': 'lgb', 'k': 2048, 'post': 'none', 'method': 'f_classif_full', 'augment': True}})\n(OUT/'configurations.json').write_text(json.dumps(CONFIGS,indent=2))\nprint('Total new configurations after refinement:',len(CONFIGS))\n", "# Freeze the previous version; do not repeat its 59-setting search.\nBASE_CONFIGS=copy.deepcopy(CONFIGS)\nBASE_PLAN=[('v4_reference',.25),('gated_s3_aug_lgb',.1875),('raw_aug_xgb_d5',.1875),('raw_lr_k2048_C0.1',.1875),('gated_syn_down_mlp_a1',.1875)]\n\ndef fit_base(idx):\n cache={};ms=[];ws=[]\n for name,w in BASE_PLAN:\n  if name=='v4_reference':ms.extend(train_legacy(idx));ws.extend([w/3]*3)\n  else:ms.append(train_new(name,idx,cache));ws.append(w)\n return ms,np.array(ws)\n", "class NewSelector:\n def __init__(self,a,target,names):self.a=a;self.y=target;self.names=names;self.cache={}\n def select(self,method,k):\n  if (method,k) in self.cache:return self.cache[(method,k)]\n  numeric=np.array([s.startswith(('numeric:','module:')) for s in self.names])\n  with warnings.catch_warnings():\n   warnings.simplefilter('ignore',RuntimeWarning)\n   global_score=np.nan_to_num(f_classif(self.a,self.y)[0],nan=0,posinf=0,neginf=0)\n   if method=='balanced':\n    orders=[]\n    for c in range(NC):\n     yes=self.y==c\n     score=np.nan_to_num(chi2(self.a,yes)[0],nan=0,posinf=0,neginf=0)\n     high=np.asarray(self.a[yes].mean(axis=0)).ravel()>np.asarray(self.a[~yes].mean(axis=0)).ravel()\n     score[~high|numeric]=0\n     orders.append(np.argsort(-score,kind='stable')[:min(k,int((score>0).sum()))])\n    order=[]\n    for rank in range(max(map(len,orders))):\n     for o in orders:\n      if rank<len(o):order.append(int(o[rank]))\n    order+=list(np.argsort(-global_score,kind='stable'))\n   else:order=list(np.argsort(-global_score,kind='stable'))\n  z=self.a.tocsc();chosen=list(np.flatnonzero(numeric));seen=set(chosen);signatures=set()\n  for j in order:\n   j=int(j)\n   if j in seen:continue\n   seen.add(j)\n   if global_score[j]<=0:continue\n   lo,hi=z.indptr[j:j+2]\n   sig=hashlib.sha256(z.indices[lo:hi].tobytes()+z.data[lo:hi].tobytes()).digest()\n   if sig in signatures:continue\n   signatures.add(sig);chosen.append(j)\n   if len(chosen)>=k:break\n  self.cache[(method,k)]=np.sort(chosen);return self.cache[(method,k)]\n\nclass SpecificityPost:\n def fit(self,a,names,target):\n  self.names=np.array(names);self.binary=np.flatnonzero([not n.startswith(('numeric:','module:')) for n in names]);self.summary=np.flatnonzero([n.startswith(('numeric:','module:')) for n in names])\n  z=(a[:,self.binary]!=0).astype(np.float32)\n  counts=np.vstack([np.asarray(z[target==c].sum(axis=0)).ravel() for c in range(NC)])\n  prior=np.bincount(target,minlength=NC)/len(target);freq=counts.sum(axis=0)\n  posterior=(counts+10*prior[:,None])/(freq+10)\n  kl=(posterior*np.log(np.maximum(posterior,1e-12)/prior[:,None])).sum(axis=0)\n  self.weights=(.25+np.sqrt(np.maximum(kl,0))*freq/(freq+20)).astype(np.float32)\n  self.scaler=MaxAbsScaler().fit(a[:,self.summary]);return self\n def transform(self,a):\n  from sklearn.preprocessing import normalize\n  block=normalize(a[:,self.binary].multiply(self.weights),norm='l2')\n  return csr32(sparse.hstack([block,self.scaler.transform(a[:,self.summary])],format='csr'))\n\nNEW={}\nfor view in ['raw','gated_syn_down_s3']:\n for method in ['global','balanced']:\n  NEW[f'{view}_{method}_lgb4096']=dict(view=view,algo='lgb',k=4096,method=method)\nfor c in [.1,.5]:NEW[f'balanced_lr_C{c}']=dict(view='raw',algo='lr',k=2048,method='balanced',C=c)\nfor c in [1.,4.]:NEW[f'specificity_lr_C{c}']=dict(view='raw',algo='lr',k=4096,method='balanced',C=c,specificity=True)\n\ndef fit_new6(name,idx,cache):\n cfg=NEW[name];view=cfg['view']\n if view not in cache:\n  enc,space,a,_=prepare_view(idx,view);cache[view]=(enc,space,a,NewSelector(a,yi[idx],space.names))\n enc,space,a,sel=cache[view];cols=sel.select(cfg['method'],cfg['k']);target=yi[idx]\n if cfg.get('specificity'):post=SpecificityPost().fit(a[:,cols],space.names[cols],target)\n else:post=ColumnPost().fit(a[:,cols],space.names[cols])\n est=estimator(cfg);tx=post.transform(a[:,cols])\n if cfg.get('augment'):\n  if '_aug' not in cache:cache['_aug']=parse_events(rare_background(X.iloc[idx],[events[i] for i in idx],SEED+777,count=10))\n  ar=enc.transform(cache['_aug'],exclude_events=[events[i] for i in idx]);ax=post.transform(space.transform(ar)[:,cols])\n  tx=csr32(sparse.vstack([tx,ax],format='csr'));target=np.concatenate([target,target]);rw=np.r_[np.ones(len(idx)),np.full(len(idx),.5)]\n else:rw=np.ones(len(idx))\n if cfg['algo']=='xgb':est.fit(tx,target,sample_weight=compute_sample_weight('balanced',target)*rw)\n elif cfg.get('augment'):est.fit(tx,target,sample_weight=rw)\n else:est.fit(tx,target)\n m=Member(name,view,enc,space,cols,post,est)\n # Column order of a specificity block changes, but raw_names always describe the vectorizer input.\n if cfg.get('specificity'):m.names=np.concatenate([m.raw_names[post.binary],m.raw_names[post.summary]])\n return m\n(OUT/'new_configurations.json').write_text(json.dumps(NEW,indent=2))\nprint('New configurations',len(NEW))\n\nNEW.update({'gated_s3_aug_global_lgb4096': {'view': 'gated_syn_down_s3', 'algo': 'lgb', 'k': 4096, 'method': 'global', 'augment': True}, 'gated_s3_aug_balanced_lgb4096': {'view': 'gated_syn_down_s3', 'algo': 'lgb', 'k': 4096, 'method': 'balanced', 'augment': True}, 'raw_aug_global_xgb4096_d3': {'view': 'raw', 'algo': 'xgb', 'k': 4096, 'method': 'global', 'depth': 3, 'augment': True}, 'raw_aug_global_xgb4096_d5': {'view': 'raw', 'algo': 'xgb', 'k': 4096, 'method': 'global', 'depth': 5, 'augment': True}})\n(OUT/'new_configurations.json').write_text(json.dumps(NEW,indent=2))\n\nprint('Total new configurations including augmentation extension:',len(NEW))\n", "def fit_plan6(idx,plan,basefit=None):\n ms=[];ws=[];cache={}\n for name,w in plan:\n  if name=='v5_reference':\n   old,ow=basefit if basefit is not None else fit_base(idx)\n   ms.extend(old);ws.extend(ow*w)\n  else:ms.append(fit_new6(name,idx,cache));ws.append(w)\n return ms,np.array(ws)\n\nV6_PLAN=[('v5_reference',.75),('gated_s3_aug_global_lgb4096',.25)]\ndef fit_v6(idx):return fit_plan6(idx,V6_PLAN)\n", "from sklearn.svm import LinearSVC\nfrom sklearn.naive_bayes import ComplementNB\nfrom sklearn.feature_selection import SelectKBest\n\ndef expand10(raw):\n t=canon(raw);m=re.fullmatch(r'([0-9]+)_([0-9]+)([A-Z]+)>([A-Z]+)',t)\n if m and int(m[2])-int(m[1])+1==len(m[3])==len(m[4]):\n  return tuple(f'{a}{int(m[1])+j}{b}' for j,(a,b) in enumerate(zip(m[3],m[4])))\n return (t,)\n\ndef kind10(t):\n if 'fs' in t:return 'frameshift'\n if t.endswith('*'):return 'stop'\n if 'delins' in t:return 'delins'\n if 'del' in t:return 'deletion'\n if 'ins' in t:return 'insertion'\n m=re.fullmatch(r'([A-Z])[0-9]+([A-Z])',t)\n if m:return 'synonymous' if m[1]==m[2] else 'substitution'\n return 'other'\n\ndef packet10(frame,ev=None):\n if ev is None:ev=parse_events(frame[genes])\n resolved=[[(g,tuple(sorted({s for t in ts for s in expand10(t)}))) for g,ts in row] for row in ev]\n missing=frame[genes].isna().to_numpy()\n return dict(events=resolved,missing=missing)\n\ndef records10(packet,representation,numeric=True):\n rs=[]\n for i,row in enumerate(packet['events']):\n  d={};syn=nonsyn=structural=unparsed=0\n  for g,ts in row:\n   d['gene:'+g]=1.;types={kind10(t) for t in ts}\n   sy='synonymous' in types;ns=bool(types.intersection({'substitution','stop','frameshift','deletion','insertion','delins'}));st=bool(types.intersection({'stop','frameshift','deletion','insertion','delins'}));ot='other' in types\n   syn+=sy;nonsyn+=ns;structural+=st;unparsed+=ot\n   if representation=='type':\n    for typ in types:d['type:'+g+':'+typ]=1.\n   elif representation=='functional':\n    for flag,family in [(sy,'syn_gene'),(ns,'nonsyn_gene'),(st,'structural_gene'),(ot,'unparsed_gene')]:\n     if flag:d[family+':'+g]=1.\n  if numeric:\n   d['numeric:log_observed_mutated_genes']=np.log1p(len(row));d['numeric:log_missing_genes']=np.log1p(int(packet['missing'][i].sum()))\n   if representation!='gene':\n    for title,value in [('syn',syn),('nonsyn',nonsyn),('structural',structural),('unparsed',unparsed)]:\n     d['numeric:log_'+title+'_genes']=np.log1p(value);d['numeric:fraction_'+title+'_genes']=value/max(len(row),1)\n  rs.append(d)\n return rs\n\ndef mask_packet10(packet,probability,seed):\n rng=np.random.default_rng(seed);out=[];missing=packet['missing'].copy();gpos={g:j for j,g in enumerate(genes)}\n for i,row in enumerate(packet['events']):\n  keep=[]\n  for g,ts in row:\n   if rng.random()<probability:missing[i,gpos[g]]=True\n   else:keep.append((g,ts))\n  out.append(keep)\n return dict(events=out,missing=missing)\n\ndef select_cols10(a,target,k):\n if k is None or a.shape[1]<=k:return np.arange(a.shape[1])\n with warnings.catch_warnings():\n  warnings.simplefilter('ignore',RuntimeWarning);scores=np.nan_to_num(chi2(a,target)[0],nan=0,posinf=0,neginf=0)\n return np.sort(np.argsort(-scores,kind='stable')[:k])\n\nclass MissingBernoulli10:\n def __init__(self,alpha=.1,pooled=False,fit_prior=False,k=None):self.alpha=alpha;self.pooled=pooled;self.fit_prior=fit_prior;self.k=k\n def fit(self,a,target,missing,sample_weight=None):\n  self.cols=select_cols10(a,target,self.k);a=csr32(a[:,self.cols]);missing=csr32(missing[:,self.cols]);w=np.ones(len(target)) if sample_weight is None else np.asarray(sample_weight)\n  self.classes_=np.unique(target);nclass=np.array([w[target==c].sum() for c in self.classes_])\n  counts=np.vstack([np.asarray(a[target==c].multiply(w[target==c,None]).sum(axis=0)).ravel() for c in self.classes_])\n  absent=np.vstack([np.asarray(missing[target==c].multiply(w[target==c,None]).sum(axis=0)).ravel() for c in self.classes_])\n  observed=nclass[:,None]-absent\n  if self.pooled:\n   denominator=w.sum()-np.asarray(missing.multiply(w[:,None]).sum(axis=0)).ravel()\n   prior=np.clip(np.asarray(a.multiply(w[:,None]).sum(axis=0)).ravel()/np.maximum(denominator,1e-9),1e-7,1-1e-7)\n   prob=(counts+self.alpha*prior)/(observed+self.alpha)\n  else:prob=(counts+self.alpha)/(observed+2*self.alpha)\n  prob=np.clip(prob,1e-7,1-1e-7);self.log0=np.log1p(-prob);self.logit=np.log(prob)-self.log0\n  prior=nclass/nclass.sum() if self.fit_prior else np.full(len(nclass),1/len(nclass));self.logprior=np.log(prior)\n  self.coef_=self.logit;return self\n def predict_proba(self,a,missing):\n  a=csr32(a[:,self.cols]);missing=csr32(missing[:,self.cols])\n  score=np.asarray(a@self.logit.T)+self.log0.sum(axis=1)+self.logprior-np.asarray(missing@self.log0.T)\n  return softmax(score,axis=1)\n\nclass GroupSigmoidSVM10:\n def __init__(self,C=.1,k=2048,balanced_calibration=False):self.C=C;self.k=k;self.balanced_calibration=balanced_calibration\n def fit(self,a,target,profile_groups,sample_weight):\n  self.classes_=np.unique(target);oof=np.zeros((len(target),len(self.classes_)),float)\n  for tr,va in StratifiedGroupKFold(3,shuffle=True,random_state=310).split(a,target,profile_groups):\n   assert not set(profile_groups[tr]).intersection(profile_groups[va])\n   cols=select_cols10(a[tr],target[tr],self.k);scaler=MaxAbsScaler().fit(a[tr][:,cols])\n   est=LinearSVC(C=self.C,class_weight='balanced',dual='auto',max_iter=5000,random_state=42).fit(csr32(scaler.transform(a[tr][:,cols])),target[tr],sample_weight=sample_weight[tr])\n   scores=est.decision_function(csr32(scaler.transform(a[va][:,cols])))\n   assert np.array_equal(est.classes_,self.classes_);oof[va]=scores\n  self.calibrators=[]\n  for j,c in enumerate(self.classes_):\n   cal=LogisticRegression(C=1.,class_weight='balanced' if self.balanced_calibration else None,max_iter=1000).fit(oof[:,j,None],(target==c).astype(int),sample_weight=sample_weight);self.calibrators.append(cal)\n  self.cols=select_cols10(a,target,self.k);self.scaler=MaxAbsScaler().fit(a[:,self.cols]);self.est=LinearSVC(C=self.C,class_weight='balanced',dual='auto',max_iter=5000,random_state=42).fit(csr32(self.scaler.transform(a[:,self.cols])),target,sample_weight=sample_weight);self.coef_=self.est.coef_;return self\n def predict_proba(self,a):\n  score=self.est.decision_function(csr32(self.scaler.transform(a[:,self.cols])))\n  p=np.column_stack([cal.predict_proba(score[:,j,None])[:,1] for j,cal in enumerate(self.calibrators)])\n  return p/np.maximum(p.sum(axis=1,keepdims=True),1e-12)\n\nclass Member10:\n def __init__(self,name,space,est,representation,numeric,cols=None,scaler=None):\n  self.name=name;self.view='v10_'+representation;self.representation=representation;self.numeric=numeric;self.vec=space.compact_vec(np.arange(len(space.names)));self.est=est;self.cols=est.cols if cols is None else cols;self.raw_names=space.names[self.cols];self.names=self.raw_names;self.scaler=scaler\n  lookup={g:i for i,g in enumerate(genes)};self.feature_gene_positions=np.array([lookup.get(str(s).split(':')[1],-1) if ':' in s and not str(s).startswith('numeric:') else -1 for s in space.names])\n def missing_matrix(self,packet):\n  out=np.zeros((len(packet['events']),len(self.feature_gene_positions)),bool);valid=self.feature_gene_positions>=0\n  out[:,valid]=packet['missing'][:,self.feature_gene_positions[valid]]\n  return csr32(sparse.csr_matrix(out))\n def predict_packet(self,packet,records=None):\n  rs=records10(packet,self.representation,self.numeric) if records is None else records;a=csr32(self.vec.transform(rs))\n  if isinstance(self.est,MissingBernoulli10):return self.est.predict_proba(a,self.missing_matrix(packet))\n  if isinstance(self.est,GroupSigmoidSVM10):return self.est.predict_proba(a)\n  tx=a[:,self.cols]\n  if self.scaler is not None:tx=csr32(self.scaler.transform(tx))\n  return self.est.predict_proba(tx)\n def predict_frame(self,frame):return self.predict_packet(packet10(frame[genes]))\n\npredict_members6=predict_members\ndef predict_members(members,weights,frame):\n old=[(m,w) for m,w in zip(members,weights) if not hasattr(m,'predict_frame')];p=predict_members6([m for m,w in old],[w for m,w in old],frame) if old else None\n packet=None\n for m,w in zip(members,weights):\n  if hasattr(m,'predict_frame'):\n   if packet is None:packet=packet10(frame[genes])\n   q=m.predict_packet(packet)*w;p=q if p is None else p+q\n return p\n\nC10={\n 'gene_nb_a0.1':dict(rep='gene',algo='nb',alpha=.1,pooled=False,fit_prior=False,k=None,mask=0.),\n 'gene_nb_a1':dict(rep='gene',algo='nb',alpha=1.,pooled=False,fit_prior=False,k=None,mask=0.),\n 'gene_nb_empirical_prior':dict(rep='gene',algo='nb',alpha=.1,pooled=False,fit_prior=True,k=None,mask=0.),\n 'gene_nb_pooled20':dict(rep='gene',algo='nb',alpha=20.,pooled=True,fit_prior=False,k=None,mask=0.),\n 'type_nb_pooled20':dict(rep='type',algo='nb',alpha=20.,pooled=True,fit_prior=False,k=4096,mask=0.),\n 'gene_lr':dict(rep='gene',algo='lr',C=.1,k=2048,mask=0.),\n 'gene_lr_mask10':dict(rep='gene',algo='lr',C=.1,k=2048,mask=.1),\n 'gene_lr_mask25':dict(rep='gene',algo='lr',C=.1,k=2048,mask=.25),\n 'type_lr':dict(rep='type',algo='lr',C=.1,k=2048,mask=0.),\n 'type_lr_mask10':dict(rep='type',algo='lr',C=.1,k=2048,mask=.1),\n 'functional_lr':dict(rep='functional',algo='lr',C=.1,k=2048,mask=0.),\n 'functional_lr_mask10':dict(rep='functional',algo='lr',C=.1,k=2048,mask=.1),\n 'gene_svm':dict(rep='gene',algo='svm',C=.1,k=2048,mask=0.),\n 'type_svm':dict(rep='type',algo='svm',C=.1,k=2048,mask=0.),\n 'functional_svm':dict(rep='functional',algo='svm',C=.1,k=2048,mask=0.),\n 'functional_svm_mask10':dict(rep='functional',algo='svm',C=.1,k=2048,mask=.1),\n 'functional_lgb':dict(rep='functional',algo='lgb',k=2048,mask=0.),\n 'functional_lgb_mask10':dict(rep='functional',algo='lgb',k=2048,mask=.1),\n 'functional_lgb_mask25':dict(rep='functional',algo='lgb',k=2048,mask=.25)}\n\ndef prepare10(idx,representation,numeric,cache):\n key=(representation,numeric)\n if key not in cache:\n  if '_clean' not in cache:cache['_clean']=packet10(X.iloc[idx],[events[i] for i in idx])\n  rs=records10(cache['_clean'],representation,numeric);space=Space();a=space.fit(rs);cache[key]=(space,a)\n return cache[key]\n\ndef train10(name,idx,cache):\n cfg=C10[name];numeric=cfg['algo'] not in ['nb','cnb'];space,a=prepare10(idx,cfg['rep'],numeric,cache);target=yi[idx];rw=np.ones(len(idx));train_groups=groups[idx]\n augmented=None\n if cfg['mask']:\n  key=('masked',cfg['mask'])\n  if key not in cache:cache[key]=mask_packet10(cache['_clean'],cfg['mask'],1042+int(cfg['mask']*100))\n  augmented=cache[key];ar=records10(augmented,cfg['rep'],numeric);a_aug=space.transform(ar);a=csr32(sparse.vstack([a,a_aug],format='csr'));target=np.r_[target,target];train_groups=np.r_[train_groups,train_groups];rw=np.r_[rw,.5*rw]/1.5\n if cfg['algo']=='nb':\n  est=MissingBernoulli10(cfg['alpha'],cfg['pooled'],cfg['fit_prior'],cfg['k'])\n  temp=Member10(name,space,est,cfg['rep'],False,cols=np.arange(a.shape[1]));missing=temp.missing_matrix(cache['_clean']);est.fit(a,target,missing,sample_weight=rw)\n  return Member10(name,space,est,cfg['rep'],False)\n if cfg['algo']=='cnb':\n  cols=select_cols10(a,target,cfg['k']);est=ComplementNB(alpha=cfg['alpha'],norm=False).fit(a[:,cols],target,sample_weight=rw)\n  return Member10(name,space,est,cfg['rep'],False,cols=cols)\n if cfg['algo']=='svm':\n  est=GroupSigmoidSVM10(cfg['C'],cfg['k'],cfg.get('balanced_calibration',False)).fit(a,target,train_groups,rw);return Member10(name,space,est,cfg['rep'],True)\n # Outer selection and scaler are fit on the original training patients only.\n cols=select_cols10(prepare10(idx,cfg['rep'],numeric,cache)[1],yi[idx],cfg['k'])\n scaler=MaxAbsScaler().fit(prepare10(idx,cfg['rep'],numeric,cache)[1][:,cols]) if cfg['algo']=='lr' else None\n tx=a[:,cols];tx=csr32(scaler.transform(tx)) if scaler is not None else csr32(tx)\n est=LogisticRegression(C=cfg['C'],class_weight='balanced',max_iter=2000) if cfg['algo']=='lr' else LGBMClassifier(n_estimators=300,learning_rate=.04,num_leaves=15,min_child_samples=20,reg_lambda=10,colsample_bytree=.85,class_weight='balanced',verbosity=-1,n_jobs=2,random_state=42)\n est.fit(tx,target,sample_weight=rw);return Member10(name,space,est,cfg['rep'],True,cols=cols,scaler=scaler)\n\n# A missing feature must contribute no Bernoulli presence or absence evidence.\ntoy=sparse.csr_matrix([[1,0],[1,0],[0,1],[0,1]],dtype=np.float32);toy_m=sparse.csr_matrix(toy.shape,dtype=np.float32)\nnbtoy=MissingBernoulli10(alpha=1.).fit(toy,np.array([0,0,1,1]),toy_m)\nassert np.allclose(nbtoy.predict_proba(sparse.csr_matrix([[1,0]],dtype=np.float32),sparse.csr_matrix([[0,0]],dtype=np.float32)),[[.9,.1]])\nassert np.allclose(nbtoy.predict_proba(sparse.csr_matrix([[0,0]],dtype=np.float32),sparse.csr_matrix([[1,1]],dtype=np.float32)),[[.5,.5]])\nassert all(kind10(t)=='synonymous' for t in expand10('1_2FS>FS'))\n(OUT/'configurations_v10.json').write_text(json.dumps(C10,indent=2));print('V10 candidates:',len(C10),'semantic checks passed')\n\nC10.update({'functional_svm_balanced': {'rep': 'functional', 'algo': 'svm', 'C': 0.1, 'k': 2048, 'mask': 0.0, 'balanced_calibration': True}, 'functional_svm_balanced_mask10': {'rep': 'functional', 'algo': 'svm', 'C': 0.1, 'k': 2048, 'mask': 0.1, 'balanced_calibration': True}, 'type_complement_nb_a0.1': {'rep': 'type', 'algo': 'cnb', 'alpha': 0.1, 'k': 4096, 'mask': 0.0}, 'type_complement_nb_a1': {'rep': 'type', 'algo': 'cnb', 'alpha': 1.0, 'k': 4096, 'mask': 0.0}, 'functional_complement_nb_a1': {'rep': 'functional', 'algo': 'cnb', 'alpha': 1.0, 'k': 4096, 'mask': 0.0}})\n(OUT/'configurations_v10.json').write_text(json.dumps(C10,indent=2))\nprint('Extended candidates:',len(C10))\n", "# P43: HGVS 2016 principles; this is a conservative dataset parser, not a full HGVS validator.\nAA16=set('ACDEFGHIKLMNPQRSTVWY');THREE16=dict(zip(['Ala','Arg','Asn','Asp','Cys','Gln','Glu','Gly','His','Ile','Leu','Lys','Met','Phe','Pro','Ser','Thr','Trp','Tyr','Val','Ter','Xaa'],list('ARNDCQEGHILKMFPSTWYV')+['*','X']))\n# Each event: key, kind, position, components(position,ref,alt,kind), flags, fs_stop_distance.\ndef parse_token16(raw):\n s=str(raw).strip();core=s[2:] if s.startswith('p.') else s;uncertain=core.startswith('(') and core.endswith(')');core=core[1:-1] if uncertain else core;core=normalize_token25(core)\n if re.match('^(?:'+'|'.join(THREE16)+')[0-9]+',core):core=re.sub('|'.join(THREE16),lambda m:THREE16[m[0]],core)\n key=('uncertain|' if uncertain else '')+core;flags=['uncertain'] if uncertain else [];pos=None;parts=();dist=None;kind='unparsed'\n def effect(a,b):\n  if a=='*' and b=='*':return 'stop_retained'\n  if a=='*':return 'stop_lost' if b in AA16 else 'ambiguous_residue'\n  if b=='*':return 'stop_gained' if a in AA16 else 'ambiguous_residue'\n  if a not in AA16 or b not in AA16:return 'ambiguous_residue'\n  return 'synonymous' if a==b else 'missense'\n m=re.fullmatch(r'([A-Z*])([1-9][0-9]*)([A-Z*])',core)\n if m:\n  a,p,b=m.groups();pos=int(p);kind=effect(a,b);parts=((pos,a,b,kind),)\n  if 'X' in [a,b]:flags.append('ambiguous_X')\n else:\n  m=re.fullmatch(r'([1-9][0-9]*)_([1-9][0-9]*)([A-Z*]+)>([A-Z*]+)',core)\n  if m:\n   start,end,ref,alt=m.groups();start=int(start);end=int(end);pos=start;span=end-start+1\n   if span>0 and span==len(ref)==len(alt) and all(a in AA16 or a=='*' for a in ref+alt):\n    parts=tuple((start+j,a,b,effect(a,b)) for j,(a,b) in enumerate(zip(ref,alt)));kind='synonymous_block' if all(a==b for a,b in zip(ref,alt)) else 'block_change';flags.append('resolved_equal_length_block')\n   else:kind='complex_range';flags.append('unresolved_range')\n  elif 'fs' in core:\n   kind='frameshift';m=re.match(r'([A-Z*]+)([1-9][0-9]*)',core);pos=int(m[2]) if m else None;m=re.search(r'fs(?:\\*|Ter)([1-9][0-9]*)$',core);dist=int(m[1]) if m else None\n  elif 'delins' in core:kind='delins'\n  elif 'del' in core:kind='deletion'\n  elif 'ins' in core:kind='insertion'\n  elif 'dup' in core:kind='duplication'\n  elif 'ext' in core:kind='extension'\n  elif core=='=':kind='no_change_unspecified'\n  elif re.fullmatch(r'[A-Z*]+[1-9][0-9]*',core):kind='incomplete';flags.append('missing_alternate')\n  if pos is None:\n   # Protein coordinate only: do not mistake fs termination distance for the affected residue.\n   m=re.match(r'(?:[A-Z*]+)?([1-9][0-9]*)',core);pos=int(m[1]) if m else None\n return (key,kind,pos,parts,tuple(flags),dist)\n\ndef parse_rows16(frame):\n cols=list(frame.columns);cache={};rows=[]\n for row in frame.itertuples(index=False,name=None):\n  mutations=[];missing=[];raw_n=0;duplicates=0\n  for gene,value in zip(cols,row):\n   if value is None or pd.isna(value) or (isinstance(value,str) and not value.strip()):missing.append(gene);continue\n   if str(value).strip()=='WT':continue\n   tokens=str(value).split();raw_n+=len(tokens);tokens=[normalize_token25(t) for t in tokens];unique=list(dict.fromkeys(tokens));duplicates+=len(tokens)-len(unique);events=[]\n   for token in unique:\n    if token not in cache:cache[token]=parse_token16(token)\n    events.append(cache[token])\n   mutations.append((gene,tuple(events)))\n  rows.append({'mutations':tuple(mutations),'missing':tuple(missing),'raw_tokens':raw_n,'duplicate_tokens':duplicates})\n return rows\n\nFIXTURES16=[('R132H','missense',132),('R132R','synonymous',132),('R213*','stop_gained',213),('*363*','stop_retained',363),('*213Q','stop_lost',213),('K16Rfs*23','frameshift',16),('1_2FS>FS','synonymous_block',1),('483_484MP>IA','block_change',483),('10_12AAA>DD','complex_range',10),('490del','deletion',490),('p.(Arg123His)','missense',123),('R123X','stop_gained',123),('A391','incomplete',391),('FS','unparsed',None),('G12delinsAA','delins',12)]\nfor token,expected_kind16,expected_pos16 in FIXTURES16:\n result=parse_token16(token);assert result[1]==expected_kind16 and result[2]==expected_pos16,(token,result)\nassert parse_token16('K16Rfs*23')[5]==23\nassert parse_token16('1_2FS>FS')[3]==((1,'F','F','synonymous'),(2,'S','S','synonymous'))\nassert 'uncertain' in parse_token16('p.(Arg123His)')[4]\nassert parse_rows16(pd.DataFrame({'G':['WT',None,'R1H R1H']}))[1]['missing']==('G',)\nassert parse_rows16(pd.DataFrame({'G':['WT',None,'R1H R1H']}))[2]['duplicate_tokens']==1\nprint('P43 annotation safety fixtures passed:',len(FIXTURES16),flush=True)\n# P25-P40: small, explicit gene subsets in the publications, not whole pathways or diagnostic rules.\nPANELS16={\n 'P25_breast':('TP53','PIK3CA','GATA3','MAP3K1'),\n 'P26_colorectal':('APC','TP53','SMAD4','PIK3CA','KRAS','ARID1A','SOX9','FAM123B','POLE','MLH1'),\n 'P27_glioma':('IDH1','IDH2','TP53','ATRX','CIC','FUBP1','NOTCH1'),\n 'P29_gbm':('EGFR','PDGFRA'),\n 'P30_renal':('VHL','PBRM1','SETD2','ARID1A','SMARCA4'),\n 'P31_papillary_renal':('MET','SETD2','FH'),\n 'P32_ovarian':('TP53','NF1','BRCA1','BRCA2','RB1','CDK12'),\n 'P33_uterine':('TP53','PTEN','CTNNB1','PIK3CA','ARID1A','KRAS','ARID5B','POLE'),\n 'P34_thyroid':('BRAF','EIF1AX','PPM1D','CHEK2'),\n 'P35_prostate':('SPOP','FOXA1','IDH1'),\n 'P36_lung_ad':('RIT1','MGA','EGFR','RBM10','NF1','MET','ERBB2'),\n 'P37_lung_scc':('TP53','HLA-A','NFE2L2','KEAP1','CDKN2A','RB1'),\n 'P38_melanoma':('BRAF','NRAS','HRAS','KRAS','NF1','KIT'),\n 'P39_gastric':('PIK3CA','RHOA'),\n 'P40_headneck':('PIK3CA','TRAF3','TP53','CDKN2A','HRAS','CASP8','NOTCH1','NSD1','AJUBA','FAT1','NFE2L2')}\nprint('Literature panel subsets:',len(PANELS16),'(external biological prior; no external patient data)',flush=True)\n", "from sklearn.feature_extraction import DictVectorizer\nfrom sklearn.feature_selection import chi2\nfrom sklearn.preprocessing import MaxAbsScaler\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.ensemble import ExtraTreesClassifier\nfrom scipy import sparse\nfrom collections import Counter\nimport copy\nVIEWS16=['legacy_raw','hierarchical','rare_backoff','rare_backoff_panels']\n# Literature-derived panel snapshots are explicit priors, not learned class labels.\nclass Preprocess16:\n def __init__(self,view,min_df=3,k=4096):self.view=view;self.min_df=min_df;self.k=k\n def fit(self,rows,target):\n  self.df=Counter((g,e[0]) for row in rows for g,evs in row['mutations'] for e in {e[0]:e for e in evs}.values())\n  records=self.records(rows,training=True);self.vec=DictVectorizer(dtype=np.float32);a=self.vec.fit_transform(records);names=self.vec.get_feature_names_out();support=np.asarray((a!=0).sum(0)).ravel();keep=(support>=self.min_df)|np.array([s.startswith(('num:','module:','numeric:','panel:')) for s in names]);self.keep=np.flatnonzero(keep);a=a[:,self.keep]\n  with warnings.catch_warnings():\n   warnings.simplefilter('ignore');scores=np.nan_to_num(chi2(a,target)[0],nan=0,posinf=0,neginf=0)\n  self.cols=np.sort(np.argsort(-scores,kind='stable')[:min(self.k,a.shape[1])]);self.names=names[self.keep[self.cols]];selected=a[:,self.cols];self.scale=MaxAbsScaler().fit(selected);self.residual_weights=np.ones(len(self.names),np.float32)\n  if self.view.startswith('rare_backoff'):\n   for j,name in enumerate(self.names):\n    if name.startswith('variant:'):\n     _,gene,key=name.split(':',2);n=self.df.get((gene,key),0);self.residual_weights[j]=n/(n+5.)\n  return self\n def records(self,rows,training=False):\n  # legacy records were computed pointwise using the frozen v6 raw encoder.\n  if self.view=='legacy_raw':return [r['legacy'] for r in rows]\n  result=[]\n  for row in rows:\n   d={};types=Counter();gene_effects={};n=0;rare=0;new=0;uncertain=0;incomplete=0;components=0;ambiguous_x=0\n   for g,evs in row['mutations']:\n    evs=list({e[0]:e for e in evs}.values());d['gene:'+g]=1.;d['gene_event_count:'+g]=np.log1p(len(evs));local=Counter();bin64=set();bin256=set()\n    for key,kind,pos,parts,flags,fsdist in evs:\n     n+=1;support=max(0,self.df.get((g,key),0)-(int(training)));israre=support<3;rare+=israre;new+=support==0;uncertain+='uncertain' in flags;incomplete+=kind in ['incomplete','unparsed','complex_range'];ambiguous_x+='ambiguous_X' in flags\n     effects=[p[3] for p in parts] or [kind];types.update(effects);local.update(effects);d['format:'+g+':'+kind]=1.;components+=len(parts)\n     if self.df.get((g,key),0)>=3:\n      d['variant:'+g+':'+key]=1.\n     if self.view.startswith('rare_backoff'):\n      bucket='rare' if israre else 'supported';d[bucket+'_gene:'+g]=d.get(bucket+'_gene:'+g,0)+1\n     for p,a,b,effect in parts:\n      d['aa_meta:'+g+':'+a+'>'+b]=d.get('aa_meta:'+g+':'+a+'>'+b,0)+1;d['aa_global:'+a+'>'+b]=d.get('aa_global:'+a+'>'+b,0)+1\n     if pos is not None:bin64.add((pos-1)//64);bin256.add((pos-1)//256)\n     # Absolute protein bins are an engineering heuristic, not genomic bins or biological domains.\n     if fsdist is not None:d['fs_stop_distance:'+g]=np.log1p(fsdist)\n    for kind,count in local.items():d['effect:'+g+':'+kind]=np.log1p(count)\n    for b in bin64:d[f'protein_bin64:{g}:{b}']=1.\n    for b in bin256:d[f'protein_bin256:{g}:{b}']=1.\n    gene_effects[g]=local\n   for name,count in [('observed_genes',len(row['mutations'])),('unique_events',n),('raw_tokens',row['raw_tokens']),('duplicate_tokens',row['duplicate_tokens']),('missing_genes',len(row['missing'])),('uncertain_events',uncertain),('incomplete_events',incomplete),('ambiguous_X',ambiguous_x),('resolved_components',components)]:d['num:log_'+name]=np.log1p(count)\n   for kind in ['synonymous','missense','stop_gained','stop_lost','stop_retained','frameshift','deletion','insertion','delins','complex_range','incomplete','unparsed','ambiguous_residue']:\n    d['num:log_'+kind]=np.log1p(types[kind]);d['num:fraction_'+kind]=types[kind]/max(sum(types.values()),1)\n   if self.view.startswith('rare_backoff'):d['num:log_rare_events']=np.log1p(rare);d['num:rare_fraction']=rare/max(n,1);d['num:log_unseen_events']=np.log1p(new)\n   if self.view.endswith('panels'):\n    observed=set(gene_effects)\n    for panel,gs in PANELS16.items():\n     count=len(observed.intersection(gs));d['panel:'+panel+':mutated_genes']=np.log1p(count)\n     flags=sum(bool(set(gene_effects.get(g,{})).intersection({'stop_gained','frameshift','deletion','delins'})) for g in gs);d['panel:'+panel+':truncation_notation_genes']=np.log1p(flags)\n   result.append(d)\n  return result\n def transform(self,rows,training=False):\n  a=self.vec.transform(self.records(rows,training=training));a=a[:,self.keep][:,self.cols];return sparse.csr_matrix(self.scale.transform(a).multiply(self.residual_weights),dtype=np.float32)\n\nclass Model16:\n def __init__(self,view,algo):self.view=view;self.algo=algo\n def fit(self,rows,target):\n  self.pre=Preprocess16(self.view).fit(rows,target);a=self.pre.transform(rows,training=True)\n  self.est=LogisticRegression(C=.1,class_weight='balanced',max_iter=1200) if self.algo=='lr' else ExtraTreesClassifier(n_estimators=200,max_depth=20,min_samples_leaf=2,max_features='sqrt',class_weight='balanced',n_jobs=2,random_state=16016)\n  self.est.fit(a,target);return self\n def predict_proba(self,rows):\n  p=self.est.predict_proba(self.pre.transform(rows));out=np.zeros((len(rows),NC),np.float64);out[:,self.est.classes_.astype(int)]=p;return out\n def predict_frame(self,frame):\n  rows=parse_rows16(frame[GENES16]);legacy=EvidenceEncoder(1.,1.).fit([]).transform(parse_events(frame[GENES16]))\n  for row,d in zip(rows,legacy):row['legacy']=d\n  return self.predict_proba(rows)\n", "# Protocol frozen before new outer results. The old holdout is never scored.\nfrom sklearn.metrics import f1_score\nfrom sklearn.utils.class_weight import compute_sample_weight\nimport gc\nGENES16=list(genes)\nparsed17=parse_rows16(X)\nraw17=EvidenceEncoder(1.,1.).fit([]).transform(parse_events(X))\nfor row,record in zip(parsed17,raw17):row['legacy']=record\n# Identical canonical and raw profiles stay in the same group.\nparent17=np.arange(len(train))\ndef root17(i):\n while parent17[i]!=i:parent17[i]=parent17[parent17[i]];i=parent17[i]\n return i\ndef join17(a,b):parent17[root17(b)]=root17(a)\ncanonical17=[hashlib.sha256(repr((tuple(sorted((g,tuple(sorted(set(e[0] for e in ev)))) for g,ev in row['mutations'])),tuple(sorted(row['missing'])))).encode()).hexdigest() for row in parsed17]\nfor identifiers in [groups,canonical17]:\n seen17={}\n for i,key in enumerate(identifiers):\n  if key in seen17:join17(seen17[key],i)\n  else:seen17[key]=i\nG17=np.array([root17(i) for i in range(len(train))]);overlap17=set(G17[dev])&set(G17[hold]);D17=dev[~np.isin(G17[dev],list(overlap17))]\nassert not set(G17[D17])&set(G17[hold])\nPLANS17={'v10_reference':(1.,0.,0.),'add_lr20':(.8,.2,0.),'add_lgb20':(.8,0.,.2),'add_both40':(.6,.2,.2)}\nPROTOCOL17={'version':17,'outer_seed':17017,'outer_folds':3,'inner_seed_base':17018,'inner_folds':2,'plans':PLANS17,'new_view':'rare_backoff','k':8192,'min_df':3,'lr_C':1.,'lgb':{'n_estimators':300,'num_leaves':15,'max_depth':5,'learning_rate':.04,'reg_lambda':10.,'min_child_samples':20},'selection':'inner mean Macro F1; baseline wins exact tie','test_opened':False,'holdout_scored':False,'adoption_gate':'selected-v10 mean >=0.003 and every outer delta>=0; else experimental only','historical_development_bias':'same development cohort used before; new seeds do not make external validation'}\nSHA17=hashlib.sha256((DATA/'train.csv').read_bytes()).hexdigest()\n# Frozen baseline parameters are embedded; no baseline search.\ndef f17(target,p):return float(f1_score(target,p.argmax(1),average='macro',labels=np.arange(NC),zero_division=0))\nclass Rich17:\n def __init__(self,algo):self.algo=algo\n def fit(self,rows,target):\n  self.pre=Preprocess16('rare_backoff',min_df=3,k=8192).fit(rows,target);a=self.pre.transform(rows,training=True)\n  if self.algo=='lr':self.est=LogisticRegression(C=1.,class_weight='balanced',max_iter=1600)\n  else:self.est=LGBMClassifier(n_estimators=300,num_leaves=15,max_depth=5,learning_rate=.04,reg_lambda=10.,min_child_samples=20,colsample_bytree=.85,class_weight='balanced',verbosity=-1,n_jobs=2,random_state=17017)\n  self.est.fit(a,target);return self\n def predict_proba(self,rows):\n  p=self.est.predict_proba(self.pre.transform(rows));out=np.zeros((len(rows),NC));out[:,self.est.classes_.astype(int)]=p;return out\n def predict_frame(self,frame):return self.predict_proba(parse_rows16(frame[GENES16]))\nclass Ensemble17:\n def __init__(self,old,weights,nb,rich,plan):self.old=old;self.weights=weights;self.nb=nb;self.rich=rich;self.plan=plan\n def predict_frame(self,frame):\n  w=PLANS17[self.plan];p=w[0]*(.75*predict_members(self.old,self.weights,frame)+.25*self.nb.predict_frame(frame))\n  for key,weight in [('lr',w[1]),('lgb',w[2])]:\n   if weight:p+=weight*self.rich[key].predict_frame(frame)\n  return p\n\n", "from sklearn.naive_bayes import ComplementNB,BernoulliNB\nimport gc\n# Separate follow-up hypothesis after v18 first-fold result: counts and absence, not another tree.\nPLANS19={'v17_reference':(1.,0.,0.),'cnb10':(.9,.1,0.),'bnb10':(.9,0.,.1),'both20':(.8,.1,.1)}\nPROTOCOL19={'version':19,'outer_seed':17017,'outer_folds':3,'inner_seed_base':17018,'inner_folds':2,'confirmation_seed':19029,'confirmation_folds':3,'plans':PLANS19,'baseline':'fixed v17 add_both40','view':'rare_backoff','k':8192,'min_df':3,'cnb':{'alpha':1.,'norm':False},'bnb':{'alpha':1.,'binarize':0.,'fit_prior':False},'probability_temperature':2.,'primary_gate':'nested mean delta>=0.003 and each fold>=0','confirmation_gate':'fixed-plan mean delta>=0.003 and each fold>=0','test_opened':False,'holdout_scored':False,'historical_bias':'adaptive follow-up after v18 first-fold result; previous primary split already observed; never pool selection with v18 outer maxima'}\n# Frozen v19 definitions only; no new v19 search.\nclass New19:\n def __init__(self,algo):self.algo=algo\n def fit(self,rows,target):\n  self.pre=Preprocess16('rare_backoff',min_df=3,k=8192).fit(rows,target);a=self.pre.transform(rows,training=True)\n  self.est=ComplementNB(alpha=1.,norm=False) if self.algo=='cnb' else BernoulliNB(alpha=1.,binarize=0.,fit_prior=False)\n  self.est.fit(a,target);return self\n def predict_proba(self,rows):\n  p=softmax(self.est.predict_log_proba(self.pre.transform(rows))/2.,axis=1);out=np.zeros((len(rows),NC));out[:,self.est.classes_.astype(int)]=p;return out\n def predict_frame(self,frame):return self.predict_proba(parse_rows16(frame[GENES16]))\nclass Ensemble19:\n def __init__(self,baseline,models,plan):self.baseline=baseline;self.models=models;self.plan=plan\n def predict_frame(self,frame):\n  w=PLANS19[self.plan];p=w[0]*self.baseline.predict_frame(frame)\n  for name,weight in [('cnb',w[1]),('bnb',w[2])]:\n   if weight:p+=weight*self.models[name].predict_frame(frame)\n  return p\n\n", "from sklearn.neural_network import MLPClassifier\nfrom sklearn.utils.class_weight import compute_sample_weight\nimport gc\n\ndef corrupt23(rows):\n # Support and nuisance pool come from the current original training fold only, without labels.\n df=Counter((g,e[0]) for row in rows for g,ev in row['mutations'] for e in {e[0]:e for e in ev}.values());pool={(g,e[0]):(g,e) for row in rows for g,ev in row['mutations'] for e in ev if df[(g,e[0])]<=2 and e[1] in ['missense','synonymous','synonymous_block']};pool=list(pool.values());assert pool;rng=np.random.default_rng(23023);aug=[]\n for row in rows:\n  mutations=row['mutations'];keep=rng.random(len(mutations))>=.1\n  if len(keep) and not keep.any():keep[rng.integers(len(keep))]=True\n  retained={g:{e[0]:e for e in ev} for (g,ev),flag in zip(mutations,keep) if flag}\n  for _ in range(int(rng.poisson(20))):\n   g,e=pool[rng.integers(len(pool))]\n   if g in row['missing']:continue\n   retained.setdefault(g,{})[e[0]]=e\n  evs=tuple((g,tuple(events23.values())) for g,events23 in retained.items());aug.append({'mutations':evs,'missing':row['missing'],'raw_tokens':sum(len(ev) for g,ev in evs),'duplicate_tokens':0})\n return aug\nclass Model23:\n def __init__(self,algo):self.algo=algo\n def fit(self,rows,target):\n  self.pre=Preprocess16('rare_backoff',min_df=3,k=8192).fit(rows,target);a=self.pre.transform(rows,training=True);aug=corrupt23(rows);at=sparse.vstack([a,self.pre.transform(aug,training=False)],format='csr');yt=np.r_[target,target]\n  if self.algo=='ova':\n   self.est=LGBMClassifier(objective='multiclassova',is_unbalance=True,n_estimators=500,learning_rate=.04,num_leaves=15,max_depth=5,min_child_samples=20,reg_lambda=10,colsample_bytree=.85,verbosity=-1,n_jobs=4,random_state=23023);sw=np.r_[np.ones(len(target)),.5*np.ones(len(target))]/1.5;self.est.fit(at,yt,sample_weight=sw)\n  else:\n   sw=compute_sample_weight('balanced',target);sw=np.r_[sw,.5*sw]/1.5;self.est=MLPClassifier(hidden_layer_sizes=(128,),alpha=1.,batch_size=128,learning_rate_init=.001,max_iter=180,early_stopping=False,random_state=23023);self.est.fit(at,yt,sample_weight=sw)\n  return self\n def predict_proba(self,rows):\n  p=np.asarray(self.est.predict_proba(self.pre.transform(rows)));p/=np.maximum(p.sum(1,keepdims=True),1e-12);out=np.zeros((len(rows),NC));out[:,self.est.classes_.astype(int)]=p;return out\n def predict_frame(self,frame):return self.predict_proba(parse_rows16(frame[GENES16]))\nPLANS23={'v19_reference':{'base':1.},'new_ova':{'ova':1.},'new_aug_mlp':{'mlp':1.},'new_average':{'ova':.5,'mlp':.5},'optional_ova20':{'base':.8,'ova':.2},'optional_mlp20':{'base':.8,'mlp':.2},'optional_both40':{'base':.6,'ova':.2,'mlp':.2}}\nPROTOCOL23={'version':23,'plans':PLANS23,'outer_seed':17017,'inner_seed_base':17018,'outer_folds':3,'inner_folds':2,'confirmation_seed':23033,'confirmation_folds':3,'preprocessor':'rare_backoff min_df3 k8192; only original fold fits vocabulary, DF, selection and scale','augmentation':{'mask_gene_prob':.1,'nuisance_poisson_mean':20,'nuisance_df_max':2,'nuisance_types':['missense','synonymous','synonymous_block'],'weight':.5,'transform_role':'inference DF on augmented rows; training LOO on originals','seed':23023},'fresh_noise_confirmation':{'poisson_mean':35,'seed_base':23034,'pool':'independent legacy rare_background mechanism, current training fold only','criterion':'mean delta>=-0.003'},'ova':{'loss':'multiclassova','is_unbalance':True,'trees':500,'leaves':15,'depth':5,'lambda':10},'mlp':{'hidden':[128],'alpha':1.,'epochs':180,'sample_weight':'balanced','early_stopping':False},'primary_gate':'mean delta>=0.005 and each fold>=-0.001','confirmation_gate':'mean delta>=0.003 and each fold>=-0.001 and paired pooled group bootstrap lower95>0 and fresh-noise35 mean delta>=-0.003','stress_gate':'historical rare-background20 and duplicate mean deltas>=-0.003','design_bias':'adaptive response to v22 historical stress decline; repeated development cohort/primary split. Fresh grouping, new draw and stronger noise are confirmation, not independent patients. No retuning on confirmation outcomes.','reference_public_user_reported':0.3664759128,'test_used_for_research':False,'old_holdout_scored':False}\n(OUT/'protocol.json').write_text(json.dumps(PROTOCOL23,indent=2))\nclass Export23:\n def __init__(self,models,plan):self.models=models;self.plan=plan;self.classes_=labels.copy();self.genes_=list(genes)\n def predict_frame(self,frame):\n  p=np.zeros((len(frame),len(self.classes_)))\n  for key,w in PLANS23[self.plan].items():p+=w*self.models[key].predict_frame(frame)\n  return p\n\ndef fit_old23(idx):\n ms,ws=fit_v6(idx);nb=train10('type_complement_nb_a0.1',idx,{});rich={key:Rich17(key).fit([parsed17[i] for i in idx],yi[idx]) for key in ['lr','lgb']};b17=Ensemble17(ms,ws,nb,rich,'add_both40');nnew=New19('cnb').fit([parsed17[i] for i in idx],yi[idx]);return Ensemble19(b17,{'cnb':nnew},'cnb10')\ndef base23(tr,va,tag):\n path=WORK/'artifacts/v19'/(tag+'.npz');meta=path.with_suffix('.json')\n if path.exists() and meta.exists():\n  info=json.loads(meta.read_text());assert info['sha']==SHA17 and info['protocol']==json.loads(json.dumps(PROTOCOL19))\n  with np.load(path) as q:\n   assert np.array_equal(q['tr'],tr) and np.array_equal(q['va'],va);return .9*q['base']+.1*q['cnb']\n old=fit_old23(tr);p=old.predict_frame(X.iloc[va]);del old;gc.collect();return p\n\ndef fit_fold23(tr,va,tag):\n assert not set(G17[tr])&set(G17[va]);assert not set(G17[tr])&set(G17[hold]);assert not set(va)&set(hold);path=OUT/(tag+'.npz');meta=path.with_suffix('.json')\n if path.exists() and meta.exists():\n  info=json.loads(meta.read_text());assert info['sha']==SHA17 and info['protocol']==json.loads(json.dumps(PROTOCOL23))\n  with np.load(path) as q:\n   assert np.array_equal(q['tr'],tr) and np.array_equal(q['va'],va);return {key:q[key] for key in q.files if key not in ['tr','va']}\n started=time.time();isconfirm=tag.startswith('confirmation');noise=None\n if isconfirm:\n  fold=int(tag.rsplit('fold',1)[1]);noise=rare_background(X.iloc[va],[events[i] for i in tr],23034+fold,count=35);old=fit_old23(tr);p={'base':old.predict_frame(X.iloc[va]),'base_noise35':old.predict_frame(noise)};del old;gc.collect()\n else:p={'base':base23(tr,va,tag)}\n print('V23 baseline ready',tag,round(time.time()-started,1),flush=True)\n for algo in ['ova','mlp']:\n  model=Model23(algo).fit([parsed17[i] for i in tr],yi[tr]);p[algo]=model.predict_proba([parsed17[i] for i in va])\n  if noise is not None:p[algo+'_noise35']=model.predict_frame(noise)\n  print('V23',tag,algo,'F1',round(f17(yi[va],p[algo]),5),'seconds',round(time.time()-started,1),flush=True);del model;gc.collect()\n for value in p.values():assert np.isfinite(value).all() and np.allclose(value.sum(1),1)\n np.savez_compressed(path,tr=tr,va=va,**p);meta.write_text(json.dumps({'sha':SHA17,'protocol':PROTOCOL23},indent=2));return p\n\ndef mix23(p,plan):return sum(w*p[key] for key,w in PLANS23[plan].items())\n", "PROTOCOL25={'plan':{'base':.6,'ova':.2,'mlp':.2}}\nclass Model25:\n\n    def __init__(self, models):\n        self.models = models\n        self.classes_ = labels.copy()\n        self.genes_ = list(genes)\n        self.training_data_sha256_ = SHA17\n\n    def predict_frame(self, frame):\n        canonical = canonical_frame25(frame.loc[:, self.genes_])\n        p = sum((w * self.models[k].predict_frame(canonical) for k, w in PROTOCOL25['plan'].items()))\n        assert np.isfinite(p).all() and np.allclose(p.sum(1), 1)\n        return p\n\ndef fit25(indices):\n    print('fresh base fit', len(indices), flush=True)\n    fitted = {'base': fit_old23(indices)}\n    for algo in ['ova', 'mlp']:\n        print('fresh', algo, 'fit', len(indices), flush=True)\n        fitted[algo] = Model23(algo).fit([parsed17[i] for i in indices], yi[indices])\n    return Model25(fitted)"]
CORE_SOURCE29="import os,sys,json,hashlib,time,gc,cloudpickle,itertools,subprocess,shutil\nfrom pathlib import Path\nimport numpy as np,pandas as pd\nfrom sklearn.model_selection import StratifiedGroupKFold\nfrom sklearn.metrics import f1_score,accuracy_score\nfrom threadpoolctl import threadpool_limits\nthreadpool_limits(2)\nroots29=[Path.cwd(),*Path.cwd().parents];REPO29=next((p for p in roots29 if (p/'data/train.csv').is_file()),None)\nif REPO29 is None:raise FileNotFoundError('data/train.csv required')\nWORK29=REPO29/'models/혜진' if (REPO29/'models/혜진').is_dir() else REPO29\nOUT29=WORK29/'artifacts/v29';OUT29.mkdir(parents=True,exist_ok=True)\ndef guard29(e,a):\n if e=='open' and a and isinstance(a[0],(str,bytes,os.PathLike)) and Path(os.fsdecode(a[0])).name in {'test.csv','sample_submission.csv'}:raise RuntimeError('v29 research cannot open test/sample')\nsys.addaudithook(guard29)\nMEMBERS29=['legacy_lr','legacy_lgb','legacy_extra','gated_s3_aug_lgb','raw_aug_xgb_d5','raw_lr_k2048_C0.1','gated_syn_down_mlp_a1','gated_s3_aug_global_lgb4096','type_cnb','rich_lr','rich_lgb','rare_cnb','aug_ova','aug_mlp']\nWEIGHTS29=dict(zip(MEMBERS29,[.0151875]*3+[.034171875]*4+[.06075,.081,.108,.108,.06,.2,.2]));assert np.isclose(sum(WEIGHTS29.values()),1)\nOLD29=set(MEMBERS29[:9]);CORE29=['rich_lr','aug_ova','aug_mlp']\nPROTOCOL29={'version':29,'purpose':'reduce classifiers, preserve feature definitions and fitted prediction rules','source':'v25 exact embedded definitions; no external pretrained model or twin rule','original_models':14,'screen_folds':3,'screen_seed':29029,'confirmation_folds':5,'confirmation_seed':29030,'cohort':'4960 repeatedly observed development patients; old1241 not evaluated; fresh partition is not independent external patients','weights':'original leaf weights renormalized; no weight grid search','screen_tolerance':{'mean_clean_delta':-.003,'worst_fold_clean_delta':-.005,'mean_noise35_delta':-.003,'worst_fold_noise35_delta':-.01},'candidates':'all 7 nonempty subsets of rich_lr/aug_ova/aug_mlp plus eligible greedy backward-removal path from all14','greedy':'each step: eligible removals only; highest clean F1, then noise F1, then fixed member order; stop when none eligible','selection':'fewest models among eligible candidates; highest clean F1 then noise F1; freeze before confirmation','confirmation_gate':{'mean_clean_delta':-.003,'worst_fold_clean_delta':-.005,'conditional_pooled_group_bootstrap_lower':-.005,'mean_noise35_delta':-.003,'worst_fold_noise35_delta':-.01},'timing':'same frame, warmed predictors, 3 alternating repetitions; excludes deserialization; latency measured separately from fit','legacy_X':'retained v25 assumption, historically motivated by test notation inspection','test_for_research':False,'public_reference':.390417094,'noninferiority_margin_is_engineering_tradeoff':True}\ndef namespace29():\n ns={'__name__':'__main__'}\n for i,s in enumerate(COMPONENT_SOURCE29):exec(compile(s,f'<v29-embedded-v25-{i}>','exec'),ns)\n return ns\ndef unpack29(model):\n b19=model.models['base'];b17=b19.baseline;members={m.name:m for m in b17.old};members.update(type_cnb=b17.nb,rich_lr=b17.rich['lr'],rich_lgb=b17.rich['lgb'],rare_cnb=b19.models['cnb'],aug_ova=model.models['ova'],aug_mlp=model.models['mlp']);assert set(members)==set(MEMBERS29)\n return members\ndef subset_weights29(names):\n total=sum(WEIGHTS29[k] for k in names);return {k:WEIGHTS29[k]/total for k in names}\nclass Compact29:\n def __init__(self,members,ns):\n  self.members=dict(members);self.weights_=subset_weights29(members);self.genes_=list(ns['genes']);self.classes_=ns['labels'].copy();self.canonical=ns['canonical_frame25'];self.parser=ns['parse_rows16'];self.predict_old=ns['predict_members'];self.training_data_sha256_=ns['SHA17']\n def predict_frame(self,frame):\n  x=self.canonical(frame.loc[:,self.genes_]);old=[k for k in self.members if k in OLD29];p=np.zeros((len(x),len(self.classes_)))\n  if old:p+=self.predict_old([self.members[k] for k in old],[self.weights_[k] for k in old],x)\n  if len(old)<len(self.members):\n   rows=self.parser(x)\n   for k,model in self.members.items():\n    if k not in OLD29:p+=self.weights_[k]*model.predict_proba(rows)\n  assert np.isfinite(p).all() and (p>=0).all() and np.allclose(p.sum(1),1);return p\ndef predict_leaves29(members,frame,ns):\n x=ns['canonical_frame25'](frame.loc[:,ns['genes']]);rows=ns['parse_rows16'](x);out=[];timings={}\n for k in MEMBERS29:\n  start=time.perf_counter();m=members[k]\n  p=ns['predict_members']([m],[1.],x) if k in OLD29 else m.predict_proba(rows)\n  assert p.shape==(len(x),len(ns['labels'])) and np.isfinite(p).all() and np.allclose(p.sum(1),1)\n  timings[k]=time.perf_counter()-start;out.append(p.astype(np.float32))\n return np.array(out),timings\ndef fit_compact29(ns,indices,names):\n members={};times={};oldcache={};newcache={};legacy=[k for k in names if k.startswith('legacy_')]\n if legacy:\n  original=ns['LEGACY'];ns['LEGACY']={k:original[k] for k in legacy};start=time.perf_counter()\n  try:\n   for m in ns['train_legacy'](indices):members[m.name]=m\n  finally:ns['LEGACY']=original\n  times['legacy_group']=time.perf_counter()-start\n for k in names:\n  if k in members:continue\n  start=time.perf_counter()\n  if k in MEMBERS29[3:7]:members[k]=ns['train_new'](k,indices,oldcache)\n  elif k=='gated_s3_aug_global_lgb4096':members[k]=ns['fit_new6'](k,indices,newcache)\n  elif k=='type_cnb':members[k]=ns['train10']('type_complement_nb_a0.1',indices,{})\n  elif k in ['rich_lr','rich_lgb']:members[k]=ns['Rich17'](k.removeprefix('rich_')).fit([ns['parsed17'][i] for i in indices],ns['yi'][indices])\n  elif k=='rare_cnb':members[k]=ns['New19']('cnb').fit([ns['parsed17'][i] for i in indices],ns['yi'][indices])\n  else:members[k]=ns['Model23']('ova' if k=='aug_ova' else 'mlp').fit([ns['parsed17'][i] for i in indices],ns['yi'][indices])\n  times[k]=time.perf_counter()-start\n return Compact29({k:members[k] for k in names},ns),times\ndef mean_score29(packs,names,role='clean'):\n weights=subset_weights29(names);scores=[]\n for pack in packs:\n  p=sum(weights[k]*pack[role][MEMBERS29.index(k)] for k in names);scores.append(float(f1_score(pack['target'],p.argmax(1),labels=np.arange(26),average='macro',zero_division=0)))\n return np.array(scores)\n"
import os,sys,json,hashlib,time,gc,cloudpickle,itertools,subprocess,shutil
from pathlib import Path
import numpy as np,pandas as pd
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import f1_score,accuracy_score
from threadpoolctl import threadpool_limits
threadpool_limits(2)
roots29=[Path.cwd(),*Path.cwd().parents];REPO29=next((p for p in roots29 if (p/'data/train.csv').is_file()),None)
if REPO29 is None:raise FileNotFoundError('data/train.csv required')
WORK29=REPO29/'models/혜진' if (REPO29/'models/혜진').is_dir() else REPO29
OUT29=WORK29/'artifacts/v29';OUT29.mkdir(parents=True,exist_ok=True)
def guard29(e,a):
 if e=='open' and a and isinstance(a[0],(str,bytes,os.PathLike)) and Path(os.fsdecode(a[0])).name in {'test.csv','sample_submission.csv'}:raise RuntimeError('v29 research cannot open test/sample')
sys.addaudithook(guard29)
MEMBERS29=['legacy_lr','legacy_lgb','legacy_extra','gated_s3_aug_lgb','raw_aug_xgb_d5','raw_lr_k2048_C0.1','gated_syn_down_mlp_a1','gated_s3_aug_global_lgb4096','type_cnb','rich_lr','rich_lgb','rare_cnb','aug_ova','aug_mlp']
WEIGHTS29=dict(zip(MEMBERS29,[.0151875]*3+[.034171875]*4+[.06075,.081,.108,.108,.06,.2,.2]));assert np.isclose(sum(WEIGHTS29.values()),1)
OLD29=set(MEMBERS29[:9]);CORE29=['rich_lr','aug_ova','aug_mlp']
PROTOCOL29={'version':29,'purpose':'reduce classifiers, preserve feature definitions and fitted prediction rules','source':'v25 exact embedded definitions; no external pretrained model or twin rule','original_models':14,'screen_folds':3,'screen_seed':29029,'confirmation_folds':5,'confirmation_seed':29030,'cohort':'4960 repeatedly observed development patients; old1241 not evaluated; fresh partition is not independent external patients','weights':'original leaf weights renormalized; no weight grid search','screen_tolerance':{'mean_clean_delta':-.003,'worst_fold_clean_delta':-.005,'mean_noise35_delta':-.003,'worst_fold_noise35_delta':-.01},'candidates':'all 7 nonempty subsets of rich_lr/aug_ova/aug_mlp plus eligible greedy backward-removal path from all14','greedy':'each step: eligible removals only; highest clean F1, then noise F1, then fixed member order; stop when none eligible','selection':'fewest models among eligible candidates; highest clean F1 then noise F1; freeze before confirmation','confirmation_gate':{'mean_clean_delta':-.003,'worst_fold_clean_delta':-.005,'conditional_pooled_group_bootstrap_lower':-.005,'mean_noise35_delta':-.003,'worst_fold_noise35_delta':-.01},'timing':'same frame, warmed predictors, 3 alternating repetitions; excludes deserialization; latency measured separately from fit','legacy_X':'retained v25 assumption, historically motivated by test notation inspection','test_for_research':False,'public_reference':.390417094,'noninferiority_margin_is_engineering_tradeoff':True}
def namespace29():
 ns={'__name__':'__main__'}
 for i,s in enumerate(COMPONENT_SOURCE29):exec(compile(s,f'<v29-embedded-v25-{i}>','exec'),ns)
 return ns
def unpack29(model):
 b19=model.models['base'];b17=b19.baseline;members={m.name:m for m in b17.old};members.update(type_cnb=b17.nb,rich_lr=b17.rich['lr'],rich_lgb=b17.rich['lgb'],rare_cnb=b19.models['cnb'],aug_ova=model.models['ova'],aug_mlp=model.models['mlp']);assert set(members)==set(MEMBERS29)
 return members
def subset_weights29(names):
 total=sum(WEIGHTS29[k] for k in names);return {k:WEIGHTS29[k]/total for k in names}
class Compact29:
 def __init__(self,members,ns):
  self.members=dict(members);self.weights_=subset_weights29(members);self.genes_=list(ns['genes']);self.classes_=ns['labels'].copy();self.canonical=ns['canonical_frame25'];self.parser=ns['parse_rows16'];self.predict_old=ns['predict_members'];self.training_data_sha256_=ns['SHA17']
 def predict_frame(self,frame):
  x=self.canonical(frame.loc[:,self.genes_]);old=[k for k in self.members if k in OLD29];p=np.zeros((len(x),len(self.classes_)))
  if old:p+=self.predict_old([self.members[k] for k in old],[self.weights_[k] for k in old],x)
  if len(old)<len(self.members):
   rows=self.parser(x)
   for k,model in self.members.items():
    if k not in OLD29:p+=self.weights_[k]*model.predict_proba(rows)
  assert np.isfinite(p).all() and (p>=0).all() and np.allclose(p.sum(1),1);return p
def predict_leaves29(members,frame,ns):
 x=ns['canonical_frame25'](frame.loc[:,ns['genes']]);rows=ns['parse_rows16'](x);out=[];timings={}
 for k in MEMBERS29:
  start=time.perf_counter();m=members[k]
  p=ns['predict_members']([m],[1.],x) if k in OLD29 else m.predict_proba(rows)
  assert p.shape==(len(x),len(ns['labels'])) and np.isfinite(p).all() and np.allclose(p.sum(1),1)
  timings[k]=time.perf_counter()-start;out.append(p.astype(np.float32))
 return np.array(out),timings
def fit_compact29(ns,indices,names):
 members={};times={};oldcache={};newcache={};legacy=[k for k in names if k.startswith('legacy_')]
 if legacy:
  original=ns['LEGACY'];ns['LEGACY']={k:original[k] for k in legacy};start=time.perf_counter()
  try:
   for m in ns['train_legacy'](indices):members[m.name]=m
  finally:ns['LEGACY']=original
  times['legacy_group']=time.perf_counter()-start
 for k in names:
  if k in members:continue
  start=time.perf_counter()
  if k in MEMBERS29[3:7]:members[k]=ns['train_new'](k,indices,oldcache)
  elif k=='gated_s3_aug_global_lgb4096':members[k]=ns['fit_new6'](k,indices,newcache)
  elif k=='type_cnb':members[k]=ns['train10']('type_complement_nb_a0.1',indices,{})
  elif k in ['rich_lr','rich_lgb']:members[k]=ns['Rich17'](k.removeprefix('rich_')).fit([ns['parsed17'][i] for i in indices],ns['yi'][indices])
  elif k=='rare_cnb':members[k]=ns['New19']('cnb').fit([ns['parsed17'][i] for i in indices],ns['yi'][indices])
  else:members[k]=ns['Model23']('ova' if k=='aug_ova' else 'mlp').fit([ns['parsed17'][i] for i in indices],ns['yi'][indices])
  times[k]=time.perf_counter()-start
 return Compact29({k:members[k] for k in names},ns),times
def mean_score29(packs,names,role='clean'):
 weights=subset_weights29(names);scores=[]
 for pack in packs:
  p=sum(weights[k]*pack[role][MEMBERS29.index(k)] for k in names);scores.append(float(f1_score(pack['target'],p.argmax(1),labels=np.arange(26),average='macro',zero_division=0)))
 return np.array(scores)


In [2]:
WORKER_SOURCE29="import sys,json,time,gc\nfrom pathlib import Path\nimport numpy as np,pandas as pd\nout=Path(sys.argv[1]);phase=sys.argv[2];fold=int(sys.argv[3]);COMPONENT_SOURCE29=json.loads((out/'component_source29.json').read_text());exec((out/'core29.py').read_text())\nns=namespace29()\nwith np.load(OUT29/f'{phase}_split{fold}.npz') as q:tr=q['tr'];va=q['va'];assert np.array_equal(ns['G17'],q['group'])\nassert not set(ns['G17'][tr])&set(ns['G17'][va]);assert not set(ns['G17'][tr])&set(ns['G17'][ns['hold']])\nstart=time.perf_counter();full=ns['fit25'](tr);full_fit=time.perf_counter()-start;members=unpack29(full);clean=ns['X'].iloc[va];noise=pd.read_pickle(OUT29/f'{phase}_noise35_fold{fold}.pkl')\nif phase=='screen':\n cp,lat=predict_leaves29(members,clean,ns);npred,nlat=predict_leaves29(members,noise,ns);flat=Compact29(members,ns)\n assert np.allclose(flat.predict_frame(clean.iloc[:12]),full.predict_frame(clean.iloc[:12]),atol=1e-7)\n assert np.allclose(sum(WEIGHTS29[k]*cp[i] for i,k in enumerate(MEMBERS29)),full.predict_frame(clean),atol=1e-6)\n assert np.allclose(flat.predict_frame(clean.iloc[:12]),flat.predict_frame(ns['legacy_frame25'](clean.iloc[:12])),atol=1e-7)\n np.savez_compressed(OUT29/f'screen_fold{fold}.npz',tr=tr,va=va,clean=cp,noise35=npred,target=ns['yi'][va],classes=ns['labels'].astype(str))\n info={'phase':phase,'fold':fold,'full_fit_seconds':full_fit,'individual_clean_predict_seconds':lat,'individual_noise_predict_seconds':nlat,'flat14_equals_original':True,'notation_probe_passed':True,'train_sha256':ns['SHA17']}\nelse:\n selection=json.loads((OUT29/'selection_frozen.json').read_text());names=selection['selected_members'];compact=Compact29({k:members[k] for k in names},ns);timed=clean.iloc[:128];compact.predict_frame(timed);full.predict_frame(timed);latfull=[];latcompact=[]\n for rep in range(3):\n  order=[('full',full),('compact',compact)] if rep%2==0 else [('compact',compact),('full',full)]\n  for name,model in order:\n   a=time.perf_counter();model.predict_frame(timed);(latfull if name=='full' else latcompact).append(time.perf_counter()-a)\n bp=full.predict_frame(clean);bn=full.predict_frame(noise);cp=compact.predict_frame(clean);cn=compact.predict_frame(noise)\n assert np.allclose(compact.predict_frame(timed.iloc[:12]),compact.predict_frame(ns['legacy_frame25'](timed.iloc[:12])),atol=1e-7)\n # Refit only the selected classifiers once, so real training cost is measured and the compact recipe is verified.\n a=time.perf_counter();standalone,times=fit_compact29(ns,tr,names);compact_fit=time.perf_counter()-a\n refit_error=float(np.max(abs(standalone.predict_frame(timed)-compact.predict_frame(timed))));assert np.allclose(standalone.predict_frame(timed),compact.predict_frame(timed),atol=1e-5),refit_error\n np.savez_compressed(OUT29/f'confirm_fold{fold}.npz',tr=tr,va=va,baseline=bp,candidate=cp,baseline_noise35=bn,candidate_noise35=cn,classes=ns['labels'].astype(str),target=ns['yi'][va])\n info={'phase':phase,'fold':fold,'train_sha256':ns['SHA17'],'selected_members':names,'full_fit_seconds':full_fit,'compact_only_fit_seconds':compact_fit,'compact_fit_breakdown':times,'full_predict_median_seconds':float(np.median(latfull)),'compact_predict_median_seconds':float(np.median(latcompact)),'timed_rows':len(timed),'repetitions':3,'standalone_refit_max_probability_error':refit_error,'notation_probe_passed':True}\n(OUT29/f'{phase}_fold{fold}.json').write_text(json.dumps(info,indent=2));print('V29 WORKER COMPLETE',phase,fold,info,flush=True)\n"
from IPython.display import display
(OUT29/'protocol_v29.json').write_text(json.dumps(PROTOCOL29,indent=2));(OUT29/'component_source29.json').write_text(json.dumps(COMPONENT_SOURCE29,ensure_ascii=False));(OUT29/'core29.py').write_text(CORE_SOURCE29);(OUT29/'worker29.py').write_text(WORKER_SOURCE29)
setup29=namespace29();D29=setup29['D17'];G29=setup29['G17'];Y29=setup29['yi'];LABELS29=setup29['labels'].astype(str);SHA29=setup29['SHA17'];assert SHA29=='92418b8441d058cfc68e939dd88725610750be4bc8edc51253cffc72fc4fc0ab'
for phase,k,seed in [('screen',3,29029),('confirm',5,29030)]:
 for fold,(a,b) in enumerate(StratifiedGroupKFold(k,shuffle=True,random_state=seed).split(setup29['X'].iloc[D29],Y29[D29],G29[D29])):
  tr,va=D29[a],D29[b];assert not set(G29[tr])&set(G29[va]);np.savez_compressed(OUT29/f'{phase}_split{fold}.npz',tr=tr,va=va,group=G29)
  path=OUT29/f'{phase}_noise35_fold{fold}.pkl'
  if not path.exists():setup29['rare_background'](setup29['X'].iloc[va],[setup29['events'][i] for i in tr],seed+1+fold,count=35).to_pickle(path)
del setup29;gc.collect()
def run_jobs29(phase,count):
 jobs=[]
 for fold in range(count):
  p=OUT29/f'{phase}_fold{fold}.npz';meta=p.with_suffix('.json')
  if p.exists() and meta.exists():
   info=json.loads(meta.read_text());assert info['train_sha256']==SHA29
   if phase=='confirm':assert info['selected_members']==json.loads((OUT29/'selection_frozen.json').read_text())['selected_members']
  else:jobs.append(fold)
 active=[]
 while jobs or active:
  while jobs and len(active)<2:
   fold=jobs.pop(0);handle=(OUT29/f'{phase}_worker{fold}.log').open('w');env=os.environ.copy();env.update(OMP_NUM_THREADS='2',OPENBLAS_NUM_THREADS='2',MKL_NUM_THREADS='2');proc=subprocess.Popen([sys.executable,str((OUT29/'worker29.py').resolve()),str(OUT29.resolve()),phase,str(fold)],cwd=REPO29,env=env,stdout=handle,stderr=subprocess.STDOUT);active.append((proc,handle,fold));print('START',phase,fold,flush=True)
  completed=[]
  for item in active:
   proc,handle,fold=item;code=proc.poll()
   if code is not None:
    handle.close()
    if code:raise RuntimeError(f'{phase} fold{fold} failed; see '+str(OUT29/f'{phase}_worker{fold}.log'))
    completed.append(item);print('COMPLETE',phase,fold,flush=True)
  active=[x for x in active if x not in completed]
  if active:time.sleep(2)
run_jobs29('screen',3)
packs=[]
for fold in range(3):
 with np.load(OUT29/f'screen_fold{fold}.npz') as q:packs.append({k:q[k].copy() for k in q.files})
baseline_clean29=mean_score29(packs,MEMBERS29);baseline_noise29=mean_score29(packs,MEMBERS29,'noise35')
candidate_rows29=[];candidate_map29={}
def evaluate29(names,label):
 names=tuple(k for k in MEMBERS29 if k in names);clean=mean_score29(packs,names);noise=mean_score29(packs,names,'noise35');dc=clean-baseline_clean29;dn=noise-baseline_noise29;t=PROTOCOL29['screen_tolerance'];ok=bool(dc.mean()>=t['mean_clean_delta'] and dc.min()>=t['worst_fold_clean_delta'] and dn.mean()>=t['mean_noise35_delta'] and dn.min()>=t['worst_fold_noise35_delta'])
 row={'label':label,'members':'|'.join(names),'count':len(names),'mean_f1':float(clean.mean()),'mean_delta':float(dc.mean()),'worst_fold_delta':float(dc.min()),'noise35_mean_delta':float(dn.mean()),'noise35_worst_delta':float(dn.min()),'eligible':ok};candidate_rows29.append(row);candidate_map29[names]=row;return row
for k in range(1,4):
 for names in itertools.combinations(CORE29,k):evaluate29(names,'core_'+str(k))
# One-at-a-time removal diagnostics, with unchanged original weights renormalized.
ablation29=[]
for name in MEMBERS29:
 row=evaluate29([k for k in MEMBERS29 if k!=name],'remove_'+name);ablation29.append({'removed':name,**row})
pd.DataFrame(ablation29).to_csv(OUT29/'leave_one_out_screen.csv',index=False)
current=tuple(MEMBERS29);path29=[]
while len(current)>1:
 trials=[]
 for name in current:
  reduced=tuple(k for k in current if k!=name);row=candidate_map29.get(reduced) or evaluate29(reduced,'greedy_remove_'+name)
  if row['eligible']:trials.append((row['mean_f1'],row['noise35_mean_delta'],-MEMBERS29.index(name),reduced,name))
 if not trials:break
 _,_,_,current,removed=max(trials,key=lambda t:t[:3]);path29.append({'removed':removed,**candidate_map29[current]})
pd.DataFrame(path29).to_csv(OUT29/'greedy_path.csv',index=False);pd.DataFrame(candidate_rows29).to_csv(OUT29/'screen_candidates.csv',index=False)
eligible=[(names,row) for names,row in candidate_map29.items() if row['eligible'] and len(names)<14]
if eligible:
 names,row=min(eligible,key=lambda item:(len(item[0]),-item[1]['mean_f1'],-item[1]['noise35_mean_delta'],item[0]));selection29={'selected_members':list(names),'selection_row':row,'screen_eligible':True}
else:selection29={'selected_members':list(MEMBERS29),'screen_eligible':False,'reason':'No reduction preserved declared clean/noise tolerance; do not invent a compact win.'}
selection29.update(weights=subset_weights29(selection29['selected_members']),baseline_screen_f1=float(baseline_clean29.mean()),protocol=PROTOCOL29);(OUT29/'selection_frozen.json').write_text(json.dumps(selection29,indent=2));print('V29 FROZEN SELECTION',json.dumps(selection29,indent=2),flush=True)
# Similarity is explanatory only; never pick members using confirmation outputs.
all_predictions=np.concatenate([p['clean'] for p in packs],axis=1);agreement=np.zeros((14,14))
for i in range(14):
 for j in range(14):agreement[i,j]=np.mean(all_predictions[i].argmax(1)==all_predictions[j].argmax(1))
pd.DataFrame(agreement,index=MEMBERS29,columns=MEMBERS29).to_csv(OUT29/'screen_prediction_agreement.csv')
del packs,all_predictions;gc.collect();run_jobs29('confirm',5)
confirmation29=[];base_oof29=np.zeros((len(D29),26));candidate_oof29=np.zeros_like(base_oof29);position29={i:j for j,i in enumerate(D29)}
for fold in range(5):
 with np.load(OUT29/f'confirm_fold{fold}.npz') as q:
  va=q['va'];bp=q['baseline'];cp=q['candidate'];bn=q['baseline_noise35'];cn=q['candidate_noise35'];target=q['target'];pos=np.array([position29[i] for i in va]);base_oof29[pos]=bp;candidate_oof29[pos]=cp
 f=lambda p:float(f1_score(target,p.argmax(1),labels=np.arange(26),average='macro',zero_division=0));info=json.loads((OUT29/f'confirm_fold{fold}.json').read_text())
 confirmation29.append({'fold':fold,'baseline_f1':f(bp),'candidate_f1':f(cp),'delta':f(cp)-f(bp),'baseline_accuracy':float(accuracy_score(target,bp.argmax(1))),'candidate_accuracy':float(accuracy_score(target,cp.argmax(1))),'noise35_delta':f(cn)-f(bn),'full_fit_seconds':info['full_fit_seconds'],'compact_fit_seconds':info['compact_only_fit_seconds'],'full_predict_seconds':info['full_predict_median_seconds'],'compact_predict_seconds':info['compact_predict_median_seconds']})
t29=pd.DataFrame(confirmation29);t29.to_csv(OUT29/'confirmation_results.csv',index=False);np.savez_compressed(OUT29/'confirmation_oof.npz',dev=D29,classes=LABELS29,baseline=base_oof29,candidate=candidate_oof29)
rng=np.random.default_rng(29031);ug=np.unique(G29[D29]);positions={g:np.flatnonzero(G29[D29]==g) for g in ug};boot=[]
for _ in range(500):
 ix=np.concatenate([positions[g] for g in rng.choice(ug,size=len(ug),replace=True)]);f=lambda p:float(f1_score(Y29[D29[ix]],p[ix].argmax(1),labels=np.arange(26),average='macro',zero_division=0));boot.append(f(candidate_oof29)-f(base_oof29))
ci=np.quantile(boot,[.025,.975]).tolist();tol=PROTOCOL29['confirmation_gate'];passed=bool(selection29['screen_eligible'] and len(selection29['selected_members'])<14 and t29.delta.mean()>=tol['mean_clean_delta'] and t29.delta.min()>=tol['worst_fold_clean_delta'] and ci[0]>tol['conditional_pooled_group_bootstrap_lower'] and t29.noise35_delta.mean()>=tol['mean_noise35_delta'] and t29.noise35_delta.min()>=tol['worst_fold_noise35_delta'])
verification29={'version':29,'original_models':14,'selected_members':selection29['selected_members'],'selected_count':len(selection29['selected_members']),'weights':selection29['weights'],'baseline_mean_f1':float(t29.baseline_f1.mean()),'candidate_mean_f1':float(t29.candidate_f1.mean()),'candidate_mean_accuracy':float(t29.candidate_accuracy.mean()),'mean_delta':float(t29.delta.mean()),'worst_fold_delta':float(t29.delta.min()),'noise35_mean_delta':float(t29.noise35_delta.mean()),'noise35_worst_delta':float(t29.noise35_delta.min()),'conditional_pooled_delta_ci95':ci,'median_predict_ratio':float(np.median(t29.compact_predict_seconds/t29.full_predict_seconds)),'median_fit_ratio':float(np.median(t29.compact_fit_seconds/t29.full_fit_seconds)),'confirmation_passed':passed,'model_saved':False,'public_score':None,'public_v25':.390417094,'test_for_research':False,'train_sha256':SHA29,'limitations':'same repeatedly observed development patients; selection and confirmation partitions differ but patients are not independent; conditional bootstrap does not include model selection/training uncertainty'}
(OUT29/'verification_v29.json').write_text(json.dumps(verification29,indent=2));print(t29.to_string(index=False),flush=True);print('V29 CONFIRMATION',json.dumps(verification29,indent=2),flush=True)


TRAIN ONLY (6201, 4386) Dev/holdout 4960 1241 OUT /Users/mac/project/cancer_project/artifacts/v29/definitions_v25
Parsed rows: 6201 Normalization examples passed
Novel settings: 44 Views: {'raw': (1.0, 1.0), 'syn_down': (1.0, 0.25), 'gated': (0.25, 1.0), 'gated_syn_down': (0.25, 0.25), 'gated_nonsyn': (0.25, 0.0)}
Total new configurations after refinement: 59
New configurations 8
Total new configurations including augmentation extension: 12
V10 candidates: 19 semantic checks passed
Extended candidates: 24
P43 annotation safety fixtures passed: 15
Literature panel subsets: 15 (external biological prior; no external patient data)
START screen 0
START screen 1
COMPLETE screen 1
START screen 2
COMPLETE screen 0
COMPLETE screen 2
V29 FROZEN SELECTION {
  "selected_members": [
    "rich_lr",
    "rich_lgb",
    "rare_cnb",
    "aug_ova"
  ],
  "selection_row": {
    "label": "greedy_remove_gated_syn_down_mlp_a1",
    "members": "rich_lr|rich_lgb|rare_cnb|aug_ova",
    "count": 4,
    "mean_f

## 4개 후보 실패 후 보수적 9개 고정 후속 실험
원래 4개 후보의 실패 판정은 유지합니다. 첫 확인 fold에서 최악 손실 기준을 넘긴 뒤, 원래 screen greedy 경로 중 기존 비중 85% 이상을 보존하는 최소 모델 수를 별도 고정했습니다. 이는 실패를 본 적응적 후속 가설이며 새 분할도 같은 환자를 사용합니다. 일반·합성 변이 허용치는 완화하지 않았습니다. 후속 seed29032 5-fold 결과로 원래 실패를 통과로 바꾸지 않습니다. 학습비용·별도 fit 동치는 첫 fold에서만 실측하고, 추론시간은 다섯 fold에서 실측합니다.

In [3]:
FOLLOWUP_SELECTION29={'selected_members': ['raw_aug_xgb_d5', 'raw_lr_k2048_C0.1', 'gated_syn_down_mlp_a1', 'type_cnb', 'rich_lr', 'rich_lgb', 'rare_cnb', 'aug_ova', 'aug_mlp'], 'weights': {'raw_aug_xgb_d5': 0.039757130651347956, 'raw_lr_k2048_C0.1': 0.039757130651347956, 'gated_syn_down_mlp_a1': 0.039757130651347956, 'type_cnb': 0.09423912450689888, 'rich_lr': 0.12565216600919848, 'rich_lgb': 0.12565216600919848, 'rare_cnb': 0.06980675889399916, 'aug_ova': 0.23268919631333057, 'aug_mlp': 0.23268919631333057}, 'retained_original_weight_mass': 0.859515625, 'screen_row': {'removed': 'gated_s3_aug_lgb', 'label': 'greedy_remove_gated_s3_aug_lgb', 'members': 'raw_aug_xgb_d5|raw_lr_k2048_C0.1|gated_syn_down_mlp_a1|type_cnb|rich_lr|rich_lgb|rare_cnb|aug_ova|aug_mlp', 'count': 9, 'mean_f1': 0.4837384559099425, 'mean_delta': 0.006120489137373601, 'worst_fold_delta': 0.0039735842583750736, 'noise35_mean_delta': 0.007211668009000416, 'noise35_worst_delta': 0.0023742528441876853, 'eligible': True}, 'protocol': {'phase': 'adaptive conservative follow-up, original 4-model result remains failed', 'reason': 'first primary confirmation fold already exceeded fixed worst-fold clean loss margin; do not relax original criterion', 'selection': 'original screen greedy path only; minimum classifiers retaining >=85% original probability weight and meeting original screen clean/noise tolerance', 'retained_mass_threshold': 0.85, 'chosen_before_followup_results': True, 'confirmation_seed': 29032, 'confirmation_folds': 5, 'noise_seed_base': 29033, 'noise_count': 35, 'same_confirmation_tolerances': True, 'runtime_benchmark': 'latency on all 5 folds; selected-only refit training benchmark and equivalence test on fold0, not all5', 'design_bias': 'adaptive response to primary failure, same development patients; not independent external confirmation', 'recorded_at_utc': '2026-11-11T05:00:10.369256+00:00', 'test_for_research': False}}
FOLLOWUP_PROTOCOL29={'phase': 'adaptive conservative follow-up, original 4-model result remains failed', 'reason': 'first primary confirmation fold already exceeded fixed worst-fold clean loss margin; do not relax original criterion', 'selection': 'original screen greedy path only; minimum classifiers retaining >=85% original probability weight and meeting original screen clean/noise tolerance', 'retained_mass_threshold': 0.85, 'chosen_before_followup_results': True, 'confirmation_seed': 29032, 'confirmation_folds': 5, 'noise_seed_base': 29033, 'noise_count': 35, 'same_confirmation_tolerances': True, 'runtime_benchmark': 'latency on all 5 folds; selected-only refit training benchmark and equivalence test on fold0, not all5', 'design_bias': 'adaptive response to primary failure, same development patients; not independent external confirmation', 'recorded_at_utc': '2026-11-11T05:00:10.369256+00:00', 'test_for_research': False}
FOLLOWUP_WORKER_SOURCE29="import sys,json,time,gc\nfrom pathlib import Path\nimport numpy as np,pandas as pd\nout=Path(sys.argv[1]);phase=sys.argv[2];fold=int(sys.argv[3]);COMPONENT_SOURCE29=json.loads((out/'component_source29.json').read_text());exec((out/'core29.py').read_text())\nns=namespace29()\nwith np.load(OUT29/f'{phase}_split{fold}.npz') as q:tr=q['tr'];va=q['va'];assert np.array_equal(ns['G17'],q['group'])\nassert not set(ns['G17'][tr])&set(ns['G17'][va]);assert not set(ns['G17'][tr])&set(ns['G17'][ns['hold']])\nstart=time.perf_counter();full=ns['fit25'](tr);full_fit=time.perf_counter()-start;members=unpack29(full);clean=ns['X'].iloc[va];noise=pd.read_pickle(OUT29/f'{phase}_noise35_fold{fold}.pkl')\nif phase=='screen':\n cp,lat=predict_leaves29(members,clean,ns);npred,nlat=predict_leaves29(members,noise,ns);flat=Compact29(members,ns)\n assert np.allclose(flat.predict_frame(clean.iloc[:12]),full.predict_frame(clean.iloc[:12]),atol=1e-7)\n assert np.allclose(sum(WEIGHTS29[k]*cp[i] for i,k in enumerate(MEMBERS29)),full.predict_frame(clean),atol=1e-6)\n assert np.allclose(flat.predict_frame(clean.iloc[:12]),flat.predict_frame(ns['legacy_frame25'](clean.iloc[:12])),atol=1e-7)\n np.savez_compressed(OUT29/f'screen_fold{fold}.npz',tr=tr,va=va,clean=cp,noise35=npred,target=ns['yi'][va],classes=ns['labels'].astype(str))\n info={'phase':phase,'fold':fold,'full_fit_seconds':full_fit,'individual_clean_predict_seconds':lat,'individual_noise_predict_seconds':nlat,'flat14_equals_original':True,'notation_probe_passed':True,'train_sha256':ns['SHA17']}\nelse:\n selection=json.loads((OUT29/'selection_followup_frozen.json').read_text());names=selection['selected_members'];compact=Compact29({k:members[k] for k in names},ns);timed=clean.iloc[:128];compact.predict_frame(timed);full.predict_frame(timed);latfull=[];latcompact=[]\n for rep in range(3):\n  order=[('full',full),('compact',compact)] if rep%2==0 else [('compact',compact),('full',full)]\n  for name,model in order:\n   a=time.perf_counter();model.predict_frame(timed);(latfull if name=='full' else latcompact).append(time.perf_counter()-a)\n bp=full.predict_frame(clean);bn=full.predict_frame(noise);cp=compact.predict_frame(clean);cn=compact.predict_frame(noise)\n assert np.allclose(compact.predict_frame(timed.iloc[:12]),compact.predict_frame(ns['legacy_frame25'](timed.iloc[:12])),atol=1e-7)\n # Refit only the selected classifiers once, so real training cost is measured and the compact recipe is verified.\n if fold==0:\n  a=time.perf_counter();standalone,times=fit_compact29(ns,tr,names);compact_fit=time.perf_counter()-a\n  refit_error=float(np.max(abs(standalone.predict_frame(timed)-compact.predict_frame(timed))));assert np.allclose(standalone.predict_frame(timed),compact.predict_frame(timed),atol=1e-5),refit_error\n else:compact_fit=None;times={'benchmark':'fold0 only, predeclared before followup results'};refit_error=None\n np.savez_compressed(OUT29/f'followup_fold{fold}.npz',tr=tr,va=va,baseline=bp,candidate=cp,baseline_noise35=bn,candidate_noise35=cn,classes=ns['labels'].astype(str),target=ns['yi'][va])\n info={'phase':phase,'fold':fold,'train_sha256':ns['SHA17'],'selected_members':names,'full_fit_seconds':full_fit,'compact_only_fit_seconds':compact_fit,'compact_fit_breakdown':times,'full_predict_median_seconds':float(np.median(latfull)),'compact_predict_median_seconds':float(np.median(latcompact)),'timed_rows':len(timed),'repetitions':3,'standalone_refit_max_probability_error':refit_error,'notation_probe_passed':True}\n(OUT29/f'{phase}_fold{fold}.json').write_text(json.dumps(info,indent=2));print('V29 WORKER COMPLETE',phase,fold,info,flush=True)\n"
def run_followup_jobs29(phase, count):
    jobs = []
    for fold in range(count):
        p = OUT29 / f'{phase}_fold{fold}.npz'
        meta = p.with_suffix('.json')
        if p.exists() and meta.exists():
            info = json.loads(meta.read_text())
            assert info['train_sha256'] == SHA29
            if phase == 'followup':
                assert info['selected_members'] == json.loads((OUT29 / 'selection_followup_frozen.json').read_text())['selected_members']
        else:
            jobs.append(fold)
    active = []
    while jobs or active:
        while jobs and len(active) < 2:
            fold = jobs.pop(0)
            handle = (OUT29 / f'{phase}_worker{fold}.log').open('w')
            env = os.environ.copy()
            env.update(OMP_NUM_THREADS='2', OPENBLAS_NUM_THREADS='2', MKL_NUM_THREADS='2')
            proc = subprocess.Popen([sys.executable, str((OUT29 / 'worker29_followup.py').resolve()), str(OUT29.resolve()), phase, str(fold)], cwd=REPO29, env=env, stdout=handle, stderr=subprocess.STDOUT)
            active.append((proc, handle, fold))
            print('START', phase, fold, flush=True)
        completed = []
        for item in active:
            proc, handle, fold = item
            code = proc.poll()
            if code is not None:
                handle.close()
                if code:
                    raise RuntimeError(f'{phase} fold{fold} failed; see ' + str(OUT29 / f'{phase}_worker{fold}.log'))
                completed.append(item)
                print('COMPLETE', phase, fold, flush=True)
        active = [x for x in active if x not in completed]
        if active:
            time.sleep(2)
primary29=json.loads((OUT29/'verification_v29.json').read_text());assert primary29['confirmation_passed'] is False
if not (OUT29/'verification_v29_primary4.json').exists():(OUT29/'verification_v29_primary4.json').write_text(json.dumps(primary29,indent=2))
for file,obj in [('followup_protocol.json',FOLLOWUP_PROTOCOL29),('selection_followup_frozen.json',FOLLOWUP_SELECTION29)]:
 path=OUT29/file
 if path.exists():assert json.loads(path.read_text())==obj
 else:path.write_text(json.dumps(obj,indent=2))
setup29=namespace29();D29=setup29['D17'];G29=setup29['G17'];Y29=setup29['yi'];LABELS29=setup29['labels'].astype(str);SHA29=setup29['SHA17']
for fold,(a,b) in enumerate(StratifiedGroupKFold(5,shuffle=True,random_state=29032).split(setup29['X'].iloc[D29],Y29[D29],G29[D29])):
 tr,va=D29[a],D29[b];assert not set(G29[tr])&set(G29[va]);np.savez_compressed(OUT29/f'followup_split{fold}.npz',tr=tr,va=va,group=G29)
 path=OUT29/f'followup_noise35_fold{fold}.pkl'
 if not path.exists():setup29['rare_background'](setup29['X'].iloc[va],[setup29['events'][i] for i in tr],29033+fold,count=35).to_pickle(path)
del setup29;gc.collect();(OUT29/'worker29_followup.py').write_text(FOLLOWUP_WORKER_SOURCE29)
run_followup_jobs29('followup',5)
rows29=[];base29=np.zeros((len(D29),26));cand29=np.zeros_like(base29);positions29={i:j for j,i in enumerate(D29)}
for fold in range(5):
 with np.load(OUT29/f'followup_fold{fold}.npz') as q:
  va=q['va'];target=q['target'];bp=q['baseline'];cp=q['candidate'];bn=q['baseline_noise35'];cn=q['candidate_noise35'];ix=np.array([positions29[i] for i in va]);base29[ix]=bp;cand29[ix]=cp
 f=lambda p:float(f1_score(target,p.argmax(1),labels=np.arange(26),average='macro',zero_division=0));info=json.loads((OUT29/f'followup_fold{fold}.json').read_text());rows29.append({'fold':fold,'baseline_f1':f(bp),'candidate_f1':f(cp),'delta':f(cp)-f(bp),'baseline_accuracy':float(accuracy_score(target,bp.argmax(1))),'candidate_accuracy':float(accuracy_score(target,cp.argmax(1))),'noise35_delta':f(cn)-f(bn),'full_fit_seconds':info['full_fit_seconds'],'compact_fit_seconds':info['compact_only_fit_seconds'],'full_predict_seconds':info['full_predict_median_seconds'],'compact_predict_seconds':info['compact_predict_median_seconds']})
table29=pd.DataFrame(rows29);table29.to_csv(OUT29/'followup_confirmation_results.csv',index=False);np.savez_compressed(OUT29/'followup_confirmation_oof.npz',dev=D29,classes=LABELS29,baseline=base29,candidate=cand29)
rng=np.random.default_rng(29034);ug=np.unique(G29[D29]);pos={g:np.flatnonzero(G29[D29]==g) for g in ug};boot=[]
for _ in range(500):
 ix=np.concatenate([pos[g] for g in rng.choice(ug,size=len(ug),replace=True)]);f=lambda p:float(f1_score(Y29[D29[ix]],p[ix].argmax(1),labels=np.arange(26),average='macro',zero_division=0));boot.append(f(cand29)-f(base29))
ci=np.quantile(boot,[.025,.975]).tolist();tol=PROTOCOL29['confirmation_gate'];passed=bool(table29.delta.mean()>=tol['mean_clean_delta'] and table29.delta.min()>=tol['worst_fold_clean_delta'] and ci[0]>tol['conditional_pooled_group_bootstrap_lower'] and table29.noise35_delta.mean()>=tol['mean_noise35_delta'] and table29.noise35_delta.min()>=tol['worst_fold_noise35_delta'])
fitted=table29.compact_fit_seconds.notna();verification29={'version':29,'phase':'adaptive conservative9 followup; original4 failed and remains failed','original_models':14,'selected_members':FOLLOWUP_SELECTION29['selected_members'],'selected_count':len(FOLLOWUP_SELECTION29['selected_members']),'weights':FOLLOWUP_SELECTION29['weights'],'selection_file':'selection_followup_frozen.json','retained_original_weight_mass':FOLLOWUP_SELECTION29['retained_original_weight_mass'],'baseline_mean_f1':float(table29.baseline_f1.mean()),'candidate_mean_f1':float(table29.candidate_f1.mean()),'candidate_mean_accuracy':float(table29.candidate_accuracy.mean()),'mean_delta':float(table29.delta.mean()),'worst_fold_delta':float(table29.delta.min()),'noise35_mean_delta':float(table29.noise35_delta.mean()),'noise35_worst_delta':float(table29.noise35_delta.min()),'conditional_pooled_delta_ci95':ci,'median_predict_ratio':float(np.median(table29.compact_predict_seconds/table29.full_predict_seconds)),'median_fit_ratio':float(np.median(table29.loc[fitted,'compact_fit_seconds']/table29.loc[fitted,'full_fit_seconds'])),'fit_benchmarked_folds':int(fitted.sum()),'confirmation_passed':passed,'primary4_confirmation_passed':False,'primary4_result':primary29,'model_saved':False,'public_score':None,'public_v25':.390417094,'test_for_research':False,'train_sha256':SHA29,'limitations':'adaptive follow-up after primary4 failure; original screen reused, fresh split but same repeatedly observed patients; conditional bootstrap excludes fitting and selection uncertainty'}
(OUT29/'verification_v29_followup9.json').write_text(json.dumps(verification29,indent=2));(OUT29/'verification_v29.json').write_text(json.dumps(verification29,indent=2));print(table29.to_string(index=False),flush=True);print('V29 CONSERVATIVE FOLLOWUP',json.dumps(verification29,indent=2),flush=True)


TRAIN ONLY (6201, 4386) Dev/holdout 4960 1241 OUT /Users/mac/project/cancer_project/artifacts/v29/definitions_v25
Parsed rows: 6201 Normalization examples passed
Novel settings: 44 Views: {'raw': (1.0, 1.0), 'syn_down': (1.0, 0.25), 'gated': (0.25, 1.0), 'gated_syn_down': (0.25, 0.25), 'gated_nonsyn': (0.25, 0.0)}
Total new configurations after refinement: 59
New configurations 8
Total new configurations including augmentation extension: 12
V10 candidates: 19 semantic checks passed
Extended candidates: 24
P43 annotation safety fixtures passed: 15
Literature panel subsets: 15 (external biological prior; no external patient data)
START followup 0
START followup 1
COMPLETE followup 1
START followup 2
COMPLETE followup 0
START followup 3
COMPLETE followup 2
START followup 4
COMPLETE followup 3
COMPLETE followup 4
 fold  baseline_f1  candidate_f1     delta  baseline_accuracy  candidate_accuracy  noise35_delta  full_fit_seconds  compact_fit_seconds  full_predict_seconds  compact_predict_seco

## 선택된 분류기만 전체 train 재학습·저장
최종 모델을 만들기 위해 제외한 분류기까지 학습하지 않습니다. 5-fold 확인 기준 미달이면 실험용으로만 저장하며 기준 모델을 대체 추천하지 않습니다.

In [4]:
verification29=json.loads((OUT29/'verification_v29.json').read_text());selection29=json.loads((OUT29/verification29.get('selection_file','selection_frozen.json')).read_text());names29=selection29['selected_members'];assert names29==verification29['selected_members'];ns29=namespace29();start29=time.perf_counter();final29,fit_times29=fit_compact29(ns29,np.arange(len(ns29['train'])),names29);total_fit29=time.perf_counter()-start29
final29.training_protocol_={'primary_protocol':PROTOCOL29,'followup_protocol':selection29.get('protocol')};final29.selected_members_=names29;final29.selection_provenance_={'screen_seed':29029,'primary_confirmation_seed':29030,'confirmation_seed':29032 if 'selection_file' in verification29 else 29030,'confirmation_passed':verification29['confirmation_passed'],'adaptive_followup':'selection_file' in verification29}
if not any(k in OLD29 for k in names29):final29.predict_old=None
path29=OUT29/'cancer_model_v29_pruned_experimental.pkl'
with path29.open('wb') as f:cloudpickle.dump(final29,f)
with path29.open('rb') as f:reloaded29=cloudpickle.load(f)
probe29=ns29['X'].iloc[:12];assert np.allclose(final29.predict_frame(probe29),reloaded29.predict_frame(ns29['legacy_frame25'](probe29)),atol=1e-7);assert np.allclose(final29.predict_frame(probe29.iloc[[8,2,0]]),final29.predict_frame(probe29)[[8,2,0]],atol=1e-6)
variable_rows29=[]
for name,member in final29.members.items():
 feature_names=getattr(getattr(member,'pre',None),'names',None)
 if feature_names is None:feature_names=getattr(member,'names',getattr(member,'raw_names',[]))
 variable_rows29.extend({'model':name,'feature':str(feature)} for feature in feature_names)
pd.DataFrame(variable_rows29).to_csv(OUT29/'selected_variables.csv',index=False)
baseline_path29=WORK29/'artifacts/v25/cancer_model_v25_legacy_stop_retrained.pkl';baseline_bytes29=baseline_path29.stat().st_size if baseline_path29.exists() else None
verification29.update(model_saved=True,model_sha256=hashlib.sha256(path29.read_bytes()).hexdigest(),model_bytes=path29.stat().st_size,baseline_model_bytes=baseline_bytes29,model_size_ratio=path29.stat().st_size/baseline_bytes29 if baseline_bytes29 else None,full_train_selected_only_fit_seconds=total_fit29,full_train_fit_breakdown=fit_times29,final_training='only retained classifiers freshly trained on all6201; removed models not trained',shared_pointwise_parsing=True)
(OUT29/'verification_v29.json').write_text(json.dumps(verification29,indent=2));print('V29 FINAL MODEL',json.dumps(verification29,indent=2),flush=True)


TRAIN ONLY (6201, 4386) Dev/holdout 4960 1241 OUT /Users/mac/project/cancer_project/artifacts/v29/definitions_v25
Parsed rows: 6201 Normalization examples passed
Novel settings: 44 Views: {'raw': (1.0, 1.0), 'syn_down': (1.0, 0.25), 'gated': (0.25, 1.0), 'gated_syn_down': (0.25, 0.25), 'gated_nonsyn': (0.25, 0.0)}
Total new configurations after refinement: 59
New configurations 8
Total new configurations including augmentation extension: 12
V10 candidates: 19 semantic checks passed
Extended candidates: 24
P43 annotation safety fixtures passed: 15
Literature panel subsets: 15 (external biological prior; no external patient data)
V29 FINAL MODEL {
  "version": 29,
  "phase": "adaptive conservative9 followup; original4 failed and remains failed",
  "original_models": 14,
  "selected_members": [
    "raw_aug_xgb_d5",
    "raw_lr_k2048_C0.1",
    "gated_syn_down_mlp_a1",
    "type_cnb",
    "rich_lr",
    "rich_lgb",
    "rare_cnb",
    "aug_ova",
    "aug_mlp"
  ],
  "selected_count": 9,
 

## 최종 고정 모델 제출 추론
연구에서 차단한 test는 마지막 별도 프로세스의 predict에만 사용합니다. 동일 입력의 배치/행 예측·확률·ID 순서를 확인합니다. Public은 미확인입니다.

In [5]:
# 별도 프로세스에서 고정 모델로 최종 test 예측만 수행합니다.
from pathlib import Path
import subprocess,sys
roots29=[Path.cwd(),*Path.cwd().parents]
repo29=next((p for p in roots29 if (p/'data/train.csv').is_file()),None)
if repo29 is None:raise FileNotFoundError('data 폴더가 필요합니다.')
work29=repo29/'models'/'혜진' if (repo29/'models'/'혜진').is_dir() else repo29
program29="import sys,os,json,hashlib,cloudpickle,numpy as np,pandas as pd\nfrom pathlib import Path\nfrom threadpoolctl import threadpool_limits\nthreadpool_limits(2)\ndef guard(e,a):\n if e=='open' and a and isinstance(a[0],(str,bytes,os.PathLike)) and Path(os.fsdecode(a[0])).name=='train.csv':raise RuntimeError('Final inference does not read train.csv')\nsys.addaudithook(guard)\nmp,tp,sp,op=map(Path,sys.argv[1:5]);proof=json.loads((mp.parent/'verification_v29.json').read_text());assert proof['model_saved']\nassert hashlib.sha256(mp.read_bytes()).hexdigest()==proof['model_sha256']\nwith mp.open('rb') as f:model=cloudpickle.load(f)\nt=pd.read_csv(tp);s=pd.read_csv(sp);assert t.ID.is_unique and s.ID.is_unique and len(t)==len(s) and set(t.ID)==set(s.ID)\nx=t.loc[:,model.genes_];p=model.predict_frame(x);assert p.shape==(len(t),26) and np.isfinite(p).all() and (p>=0).all() and np.allclose(p.sum(1),1)\nidx=[10,3,500,0];assert np.allclose(model.predict_frame(x.iloc[idx]),p[idx],atol=1e-6)\ns.SUBCLASS=s.ID.map(pd.Series(model.classes_[p.argmax(1)],index=t.ID));assert not s.isna().any().any()\nop.parent.mkdir(parents=True,exist_ok=True)\npayload=s.to_csv(index=False).encode()\nif op.exists():assert op.read_bytes()==payload,'Existing v29 submission differs; preserve it.'\nelse:op.write_bytes(payload)\nassert pd.read_csv(op).ID.equals(s.ID)\nreport={'version':29,'rows':len(s),'model_sha256':proof['model_sha256'],'csv_sha256':hashlib.sha256(op.read_bytes()).hexdigest(),'phase':'final inference only','model_fitting':False,'test_used_for_validation':False,'public_score':None,'batch_invariance':True}\n(mp.parent/'submission_verification.json').write_text(json.dumps(report,ensure_ascii=False,indent=2));print(json.dumps(report,ensure_ascii=False,indent=2))\n"
args29=[str(work29/'artifacts/v29/cancer_model_v29_pruned_experimental.pkl'),str(repo29/'data/test.csv'),str(repo29/'data/sample_submission.csv'),str(repo29/'submission/혜진_submission_v29_pruned_experimental.csv')]
run29=subprocess.run([sys.executable,'-c',program29,*args29],capture_output=True,text=True)
if run29.returncode:raise RuntimeError(run29.stderr)
print(run29.stdout)


{
  "version": 29,
  "rows": 2546,
  "model_sha256": "797df55e5abc9c3af196d4be3eed6e0d4c255f79d985ad2f22f148088f3e8e07",
  "csv_sha256": "7e9f850a569ac1cf3d7836311b1e79b61b66d463122c09e9061c1b3a49d121ed",
  "phase": "final inference only",
  "model_fitting": false,
  "test_used_for_validation": false,
  "public_score": null,
  "batch_invariance": true
}